# 🏦 Mutual Fund Prospectus Extraction System — Complete Pipeline

## Architecture Overview

This notebook walks through the **entire end-to-end pipeline** for extracting structured information from Mutual Fund prospectus PDFs. Every output is printed to the terminal so you can inspect each step.

### Pipeline Stages

| # | Stage | What it does |
|---|-------|-------------|
| 1 | **Configuration** | API keys, model settings, prompts |
| 2 | **Document Validation** | Vision model scans first N pages, scores ±10 per page (threshold ≥ 65) |
| 3 | **Dynamic Section Discovery** | Vision model scans **every** page with LangChain sliding-window memory (last 3 pages), discovers sections/subsections/keywords |
| 4 | **Document Processing** | Docling converts PDF → structured chunks with bounding boxes |
| 5 | **RAG Indexing** | Chunks indexed into ChromaDB with OpenAI embeddings |
| 6 | **Section Extraction** | RAG retrieval → LLM re-ranking → Section-level summarization |
| 7 | **Chatbot Q&A** | Ask questions about the document with re-ranked context |

### Key Technologies
- **OpenAI GPT-4o-mini** (text + vision)
- **Docling** (PDF parsing & chunking)
- **ChromaDB** (vector store)
- **LangChain** (LLM orchestration, sliding-window memory)

## 1 · Configuration

All settings live in one place: API keys, model names, chunking params, prompts.

- **No hardcoded sections** — sections are discovered dynamically by the vision model
- Three new prompts for validation, page scanning, and consolidation
- Sliding-window memory keeps last 3 page analyses for context continuity

In [ ]:
# =============================================================================
# CONFIGURATION — API keys, models, chunking, validation, and all prompts
# =============================================================================
# This cell defines every tunable parameter for the entire pipeline.
# All downstream classes read from these module-level constants so you
# only need to change values here — no hunting through scattered files.
# =============================================================================

import os

# ── OpenAI API Key ──────────────────────────────────────────────────────────
# Your OpenAI project key. It is also pushed into the env var so that
# LangChain / ChromaDB clients can pick it up automatically without
# explicitly passing the key to every constructor.
OPENAI_API_KEY = ""
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# ── ChromaDB settings ──────────────────────────────────────────────────────
# ChromaDB is our vector database. We persist it to disk so that indexed
# embeddings survive kernel restarts and we don't re-embed unnecessarily.
# COLLECTION_NAME lets you maintain separate vector spaces per document.
CHROMA_PERSIST_DIR = "./chroma_db"           # Directory where vectors are stored on disk
COLLECTION_NAME    = "mf_prospectus"         # Logical collection name inside ChromaDB

# ── Embedding model (OpenAI) ──────────────────────────────────────────────
# text-embedding-3-small is fast, cheap, and generates 1536-dim vectors.
# Alternative: "text-embedding-3-large" for higher quality at 2× cost.
EMBEDDING_MODEL = "text-embedding-3-small"

# ── LLM settings ──────────────────────────────────────────────────────────
# GPT-4o-mini is used for ALL text-only tasks:
#   - Chatbot Q&A answers
#   - Re-ranking chunks by relevance (scoring 0-10)
#   - Section extraction / summarisation
#   - Consolidating page-level scan results into a section tree
# Temperature 0.1 keeps outputs nearly deterministic — important for
# consistent re-ranking scores and factual extraction.
LLM_MODEL       = "gpt-4o-mini"             # Text LLM for chat, re-rank, extraction
LLM_TEMPERATURE = 0.1                       # Low temp → deterministic, factual answers

# ── Vision model (GPT-4o-mini supports image inputs) ─────────────────────
# The same GPT-4o-mini model also accepts images. We use it to:
#   1. Classify pages as "mutual fund content" or "not" (validation)
#   2. Extract section headings and structure from page images (discovery)
# A separate constant keeps the option open to swap to a different model
# (e.g., gpt-4o for higher-quality vision at higher cost).
VISION_MODEL = "gpt-4o-mini"

# ── Chunking settings (Docling HybridChunker) ────────────────────────────
# MAX_TOKENS_PER_CHUNK controls how big each semantic chunk can be.
# 512 tokens ≈ 350-400 words — a good balance between having enough
# context in each chunk and keeping retrieval precise.
# TOKENIZER_MODEL is the HuggingFace tokenizer used to count tokens.
MAX_TOKENS_PER_CHUNK = 512
TOKENIZER_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# ── Re-ranker settings ───────────────────────────────────────────────────
# Two-stage retrieval:
#   Stage 1: Retrieve RERANK_TOP_K chunks by vector similarity (fast, broad)
#   Stage 2: LLM scores each chunk 0-10 → keep only FINAL_TOP_K (accurate)
# This lets us cast a wide net first, then surgically pick the best chunks.
RERANK_TOP_K = 10   # How many chunks to retrieve initially (stage 1)
FINAL_TOP_K  = 5    # How many to keep after LLM re-ranking (stage 2)

# ── Document validation settings ─────────────────────────────────────────
# Before processing, we check whether the uploaded PDF is actually a
# mutual fund document using a scoring system:
#
#   1. Start at VALIDATION_INITIAL_SCORE (50 points)
#   2. Render the first VALIDATION_MAX_PAGES pages to PNG images
#   3. Vision model classifies each page:
#      - If page IS mutual fund content → add VALIDATION_SCORE_DELTA (+10)
#      - If page is NOT mutual fund content → subtract VALIDATION_SCORE_DELTA (-10)
#   4. Final score ≥ VALIDATION_THRESHOLD (65) → document accepted
#
# Example: 5 MF pages → 50 + 5×10 = 100 (pass ✅)
# Example: 3 MF + 2 non-MF → 50 + 30 − 20 = 60 (fail ❌)
VALIDATION_INITIAL_SCORE = 50    # Starting score before scanning any page
VALIDATION_THRESHOLD     = 65    # Minimum score required to accept the document
VALIDATION_SCORE_DELTA   = 10    # Points added (MF) or subtracted (non-MF) per page
VALIDATION_MAX_PAGES     = 5     # Max pages to scan (for efficiency — first 5 is enough)

# ── Sliding-window memory (for section discovery) ────────────────────────
# When scanning pages one by one for section discovery, we give the model
# "memory" of what it saw on the previous N pages. This helps it:
#   - Recognise sections that span multiple pages (continuations)
#   - Avoid duplicating sections it already found
#   - Understand the overall document flow
# Implemented via LangChain HumanMessage/AIMessage pairs — see SlidingWindowMemory.
PAGE_MEMORY_WINDOW = 3           # Keep context of last 3 pages in the sliding window

# ── Print a quick summary of all settings ────────────────────────────────
print("✅ Configuration loaded")
print(f"   LLM Model        : {LLM_MODEL}")
print(f"   Vision Model     : {VISION_MODEL}")
print(f"   Embedding Model  : {EMBEDDING_MODEL}")
print(f"   Validation score : start={VALIDATION_INITIAL_SCORE}, threshold={VALIDATION_THRESHOLD}, delta=±{VALIDATION_SCORE_DELTA}")
print(f"   Memory window    : {PAGE_MEMORY_WINDOW} pages")

In [ ]:
# =============================================================================
# PROMPTS — All prompt templates used across the pipeline
# =============================================================================
# We keep every prompt in one place for easy tweaking and version control.
# Each prompt is a string template. Some use Python's .format() for variable
# injection (double braces {{ }} escape literal braces inside .format strings).
# =============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# PROMPT 1: DOCUMENT VALIDATION (Vision)
# ─────────────────────────────────────────────────────────────────────────────
# Purpose : Classify a single PDF page image as "mutual fund content" or not.
# Called by: DocumentScanner.validate_document()
# Input   : One base64-encoded PNG image of a PDF page (sent via vision API).
# Output  : JSON with boolean "is_mf_document", confidence, and reason.
# Notes   : This prompt is sent as-is — NOT used with .format() because it
#           contains no Python placeholders. The page image is attached
#           separately as an image_url in the LangChain HumanMessage.

DOCUMENT_VALIDATION_PROMPT = """You are a financial document classifier.

Look at this PDF page image and determine whether it belongs to a
**mutual fund prospectus, factsheet, scheme information document (SID),
or key information memorandum (KIM)**.

Indicators of a mutual fund document:
- Fund name, AMC (Asset Management Company) name
- NAV, AUM, expense ratio, benchmark mentions
- Investment objective, asset allocation, portfolio holdings
- SEBI registration, regulatory disclaimers
- Risk-o-meter, riskometer
- SIP details, entry/exit load
- Performance/returns data
- Fund manager information

Respond with EXACTLY this JSON (no markdown fences):
{"is_mf_document": true, "confidence": "high", "reason": "brief explanation"}
or
{"is_mf_document": false, "confidence": "high", "reason": "brief explanation"}"""

# ─────────────────────────────────────────────────────────────────────────────
# PROMPT 2: PAGE SECTION SCAN (Vision + Sliding-Window Memory Context)
# ─────────────────────────────────────────────────────────────────────────────
# Purpose : Extract all section headings and structure from a single page image.
# Called by: DocumentScanner.discover_sections() — once per page.
# Input   : Page image + text context from the previous N pages (sliding window).
# Placeholders (filled at call time via .format()):
#   {memory_context} — text summary of the last 3 pages (from SlidingWindowMemory)
#   {page_num}       — 1-based page number
# Output  : JSON with page_number, list of sections_found (title, description,
#           keywords, subsections, is_continuation), and page_summary.
# Note    : Double braces {{ }} are used so .format() doesn't interpret them
#           as Python placeholders — they become literal { } in the final string.

PAGE_SECTION_SCAN_PROMPT = """You are an expert financial document analyst.

Analyze this PDF page image and identify all **sections and subsections**
visible on the page.

{memory_context}

INSTRUCTIONS:
1. Identify every distinct section / heading / subsection on this page
2. For each section determine:
   - The title as it appears on the page
   - A brief description of what information it contains
   - Key topics, terms, or data points (keywords for search)
   - Any subsection headings nested under it
   - Whether it is a continuation from a previous page
3. Capture tables, charts, headers, footers, disclaimers
4. If a section from a previous page continues here, note that

Respond with EXACTLY this JSON (no markdown fences):
{{
    "page_number": {page_num},
    "sections_found": [
        {{
            "title": "Section title as it appears",
            "description": "What information this section contains",
            "keywords": ["keyword1", "keyword2", "keyword3"],
            "subsections": ["Sub-heading 1", "Sub-heading 2"],
            "is_continuation": false,
            "content_summary": "One-line summary of this section on this page"
        }}
    ],
    "page_summary": "One-line summary of what this entire page covers"
}}"""

# ─────────────────────────────────────────────────────────────────────────────
# PROMPT 3: SECTION CONSOLIDATION (Text LLM — no vision needed)
# ─────────────────────────────────────────────────────────────────────────────
# Purpose : After scanning ALL pages one by one, we have N separate page-level
#           analyses. This prompt asks a text LLM to merge/deduplicate them
#           into a single hierarchical section tree for the whole document.
# Called by: DocumentScanner._consolidate_sections()
# Input   : Concatenated text of all page-level scan results.
# Placeholder: {page_analyses} — the formatted text block of all page findings.
# Output  : JSON with list of consolidated sections, each having:
#           section_key, title, description, subsections, keywords, pages.
# Why needed: A single section (e.g., "Risk Factors") might appear on pages
#           3, 4, and 5. This prompt merges those into one entry.

SECTION_CONSOLIDATION_PROMPT = """You are an expert financial document analyst.

Below are page-by-page analyses of a mutual fund document.
Consolidate them into a clean, deduplicated section/subsection structure.

PAGE-BY-PAGE FINDINGS:
{page_analyses}

INSTRUCTIONS:
1. Merge sections that span multiple pages into single entries
2. Group related subsections under their parent sections
3. Remove duplicates
4. Create a clean hierarchical structure
5. For each section compile ALL relevant keywords from all pages
6. Generate a section_key (lowercase, underscores, no special chars)
7. Order sections logically as they appear in the document

Respond with EXACTLY this JSON (no markdown fences):
{{
    "sections": [
        {{
            "section_key": "unique_key_like_this",
            "title": "Section Title",
            "description": "What this section covers",
            "subsections": ["Subsection 1", "Subsection 2"],
            "keywords": ["keyword1", "keyword2"],
            "pages": [1, 2]
        }}
    ]
}}"""

# ─────────────────────────────────────────────────────────────────────────────
# PROMPT 4: SECTION EXTRACTION (Text LLM — post-RAG summarisation)
# ─────────────────────────────────────────────────────────────────────────────
# Purpose : After RAG retrieves relevant chunks for a section, this prompt
#           asks the LLM to summarise and organise those chunks into a
#           structured report with subsection breakdowns.
# Called by: SectionExtractor.extract_section()
# Placeholders:
#   {section_title}       — e.g., "Investment Objective"
#   {section_description} — what the section covers
#   {subsections}         — comma-separated subsection titles to look for
#   {chunks}              — the actual text chunks retrieved by RAG
# Output  : Free-form structured text summary organised by subsection.

SECTION_EXTRACTION_PROMPT = """You are an expert financial analyst specializing in mutual fund prospectus analysis.

Given the following chunks from a mutual fund prospectus document, extract information relevant to the section: "{section_title}"

Section Description: {section_description}
Subsections to look for: {subsections}

CHUNKS:
{chunks}

INSTRUCTIONS:
1. Extract all relevant information for this section from the provided chunks
2. Organize the information under appropriate subsections
3. If information for a subsection is not found, indicate "Not found in document"
4. Be precise and quote exact figures when available (percentages, amounts, dates)
5. Maintain the original meaning - do not interpret or add assumptions

OUTPUT FORMAT:
Provide a structured summary with:
- Main findings for the section
- Details for each subsection found
- Key figures and data points
- Any important notes or caveats

YOUR RESPONSE:"""

# ─────────────────────────────────────────────────────────────────────────────
# PROMPT 5: CHAT SYSTEM PROMPT (Text LLM — conversational Q&A)
# ─────────────────────────────────────────────────────────────────────────────
# Purpose : Powers the chatbot Q&A feature. The user asks a question,
#           we retrieve + re-rank relevant chunks, then inject them as
#           context for the LLM to answer from.
# Called by: RAGEngine.chat()
# Placeholders:
#   {context} — formatted text of re-ranked chunks (top 5)
#   {query}   — the user's natural-language question
# Key rules: The LLM must ONLY answer from the provided chunks — no
#           hallucination. If info isn't in the chunks, say so.

CHAT_SYSTEM_PROMPT = """You are an expert financial advisor assistant helping users understand mutual fund prospectus documents.

You have access to the following document chunks that were retrieved based on the user's query.

IMPORTANT GUIDELINES:
1. Answer ONLY based on the information in the provided document chunks
2. If the information is not in the chunks, say "I couldn't find this information in the document"
3. Be precise with numbers, percentages, and dates
4. Cite specific sections or pages when possible
5. Explain financial terms if the user seems unfamiliar
6. Be helpful but never make up information

DOCUMENT CHUNKS:
{context}

USER QUERY: {query}

YOUR RESPONSE:"""

# ─────────────────────────────────────────────────────────────────────────────
# PROMPT 6: RE-RANK PROMPT (Text LLM — chunk relevance scoring)
# ─────────────────────────────────────────────────────────────────────────────
# Purpose : Given a user query and a list of candidate chunks (from
#           similarity search), the LLM scores each chunk 0-10 for
#           relevance. This is the "re-rank" step that improves
#           retrieval quality beyond pure vector similarity.
# Called by: RAGEngine.rerank()
# Placeholders:
#   {query}  — the search query (user question or section title+desc)
#   {chunks} — formatted list of chunk texts (max 500 chars each)
# Output  : JSON array of {chunk_id, score} objects.

RERANK_PROMPT = """Given a query and a list of document chunks, rate how relevant each chunk is to answering the query.

Query: {query}

For each chunk, provide a relevance score from 0 to 10:
- 10: Directly answers the query with specific information
- 7-9: Highly relevant, contains useful related information
- 4-6: Somewhat relevant, may have partial information
- 1-3: Marginally relevant
- 0: Not relevant at all

Chunks to evaluate:
{chunks}

Return your response as a JSON array of objects with 'chunk_id' and 'score' fields.
Example: [{{"chunk_id": 0, "score": 8}}, {{"chunk_id": 1, "score": 3}}, ...]

YOUR RESPONSE (JSON only):"""

# ── Print summary of all defined prompts ─────────────────────────────────
print("✅ All 6 prompts defined")
print("   1. DOCUMENT_VALIDATION_PROMPT   — classify page as MF doc or not (vision)")
print("   2. PAGE_SECTION_SCAN_PROMPT     — discover sections per page (vision + memory)")
print("   3. SECTION_CONSOLIDATION_PROMPT — merge page analyses into section tree (text)")
print("   4. SECTION_EXTRACTION_PROMPT    — summarize RAG chunks for a section (text)")
print("   5. CHAT_SYSTEM_PROMPT           — chatbot Q&A with context (text)")
print("   6. RERANK_PROMPT                — LLM-based chunk re-ranking (text)")

## 2 · Document Scanner — Vision-based Validation & Section Discovery

Two classes:

1. **`SlidingWindowMemory`** — Stores LangChain `HumanMessage` / `AIMessage` pairs. Returns last *N* AI responses as text context so the vision model can see continuity across pages.

2. **`DocumentScanner`** — Two main methods:
   - `validate_document(pdf_path)` → renders pages to PNG → sends to vision LLM → scores ±10 → returns `(is_valid, score, summary)`
   - `discover_sections(pdf_path)` → scans every page with memory context → consolidates into deduplicated section tree

In [ ]:
# =============================================================================
# SLIDING WINDOW MEMORY — LangChain-based context window for page scanning
# =============================================================================
# When the vision model scans pages one at a time during section discovery,
# it needs to know what it already found on previous pages. Otherwise it
# might duplicate sections or miss continuations across page breaks.
#
# This class maintains a sliding window of the last N page analyses using
# LangChain's HumanMessage / AIMessage objects. Each page scan creates
# one exchange (Human asks "scan page X", AI responds with what it found).
#
# get_context_string() extracts only the AI responses (the actual findings)
# from the last `window_size` exchanges and formats them into a text block
# that gets injected into the PAGE_SECTION_SCAN_PROMPT via {memory_context}.
# =============================================================================

import json                          # For parsing LLM JSON responses
import base64                        # For encoding PNG images to base64 strings
from typing import List, Dict, Tuple # Type hints for better code documentation

import fitz  # PyMuPDF — renders PDF pages to raster images (PNG)
             # Used for: (1) page rendering → vision model, (2) page count

from langchain_openai import ChatOpenAI         # LangChain wrapper for OpenAI chat models
from langchain_core.messages import HumanMessage, AIMessage  # Message types for memory


class SlidingWindowMemory:
    """
    LangChain-based sliding window memory for page-by-page document scanning.
    
    The "sliding window" concept:
    - We process pages sequentially (page 1, 2, 3, …)
    - After each page, we store what the model found (as an AIMessage)
    - When processing page N, we include the findings from pages N-3 to N-1
      (or however many fit in the window) so the model has context
    - Old pages beyond the window are dropped to keep the prompt manageable
    
    Why LangChain messages?
    - HumanMessage/AIMessage are the standard chat message types in LangChain
    - They could be fed directly into a ChatOpenAI model if needed
    - They provide a clean interface for storing conversation-like exchanges
    """

    def __init__(self, window_size: int = 3):
        """
        Args:
            window_size: Number of recent page exchanges to keep.
                         3 pages × ~500 tokens each ≈ 1500 tokens of context.
        """
        self.messages: List = []          # Flat list of LangChain message objects
        self.window_size: int = window_size  # How many page exchanges to retain

    def add_exchange(self, user_input: str, ai_output: str):
        """
        Record one page-analysis exchange.
        
        Args:
            user_input: What we asked (e.g., "Scan page 3")
            ai_output: What the model found (e.g., "Page 3: Risk Factors, ...")
            
        Each exchange = 2 messages (HumanMessage + AIMessage), so
        the list grows by 2 per page scanned.
        """
        self.messages.append(HumanMessage(content=user_input))
        self.messages.append(AIMessage(content=ai_output))

    def get_context_string(self) -> str:
        """
        Build a text block from recent AI responses for prompt injection.
        
        Returns:
            A formatted string like:
            "CONTEXT FROM PREVIOUS PAGES:
             Page 1: Fund overview, NAV, AUM...
             Page 2: Investment objective, asset allocation..."
            
            Returns empty string if no pages have been scanned yet.
        
        Implementation:
            - Each exchange = 2 messages, so last N exchanges = last N*2 messages
            - We filter for AIMessage only (skip HumanMessage)
            - Join all AI responses with newlines
        """
        # Slice the last N exchanges (each exchange = Human + AI = 2 messages)
        recent = self.messages[-(self.window_size * 2):]
        if not recent:
            return ""  # No context yet — first page being scanned
        
        # Extract only the AI responses (the actual page analysis results)
        parts = []
        for msg in recent:
            if isinstance(msg, AIMessage):
                parts.append(msg.content)
        if not parts:
            return ""
        
        # Format as a context block that goes into {memory_context} placeholder
        return "CONTEXT FROM PREVIOUS PAGES:\n" + "\n".join(parts)

    def get_messages(self):
        """
        Return the raw LangChain messages in the current window.
        Useful for debugging or if you want to feed them into a chat model.
        """
        return self.messages[-(self.window_size * 2):]


# ── Verify the class is ready ────────────────────────────────────────────
print("✅ SlidingWindowMemory class defined")
print(f"   Window size: {PAGE_MEMORY_WINDOW} pages (configurable via PAGE_MEMORY_WINDOW)")
print("   Stores HumanMessage/AIMessage pairs from LangChain")
print("   get_context_string() → text injected into PAGE_SECTION_SCAN_PROMPT")

In [ ]:
# =============================================================================
# DOCUMENT SCANNER — Vision-based validation + dynamic section discovery
# =============================================================================
# This is the core innovation of our pipeline. Instead of hardcoding which
# sections a mutual fund document should have (like the old MF_SECTIONS dict),
# we use GPT-4o-mini's vision capabilities to:
#
#   1. VALIDATE: Is this PDF actually a mutual fund document?
#      - Renders first N pages as PNG images
#      - Vision model classifies each page as MF content or not
#      - Accumulates a score: +10 per MF page, −10 per non-MF page
#      - Document passes if final score ≥ 65
#
#   2. DISCOVER: What sections/subsections does this document contain?
#      - Renders EVERY page as a PNG image
#      - Vision model extracts section headings from each page image
#      - Sliding-window memory provides context from previous pages
#      - Text LLM consolidates page-level findings into a single tree
#      - Result: a dict of {section_key: {title, description, subsections, keywords, pages}}
#
# This approach is document-agnostic — it works on any mutual fund document
# regardless of format, AMC, or regulatory framework.
# =============================================================================


class DocumentScanner:
    """
    Vision-based document scanner with two main capabilities:
    
    1. validate_document(pdf_path) → (is_valid, score, summary)
       Determines if the PDF is a genuine mutual fund document.
    
    2. discover_sections(pdf_path) → {section_key: section_config}
       Finds all sections/subsections without any hardcoded knowledge.
    
    Architecture:
        - Uses TWO LLM instances:
          • vision_llm: GPT-4o-mini with image input (for page classification & scanning)
          • text_llm: GPT-4o-mini text-only (for consolidating scan results)
        - Uses SlidingWindowMemory for cross-page context during discovery
        - Uses PyMuPDF (fitz) for rendering PDF pages to PNG images
    """

    def __init__(self):
        # ── Vision LLM ──────────────────────────────────────────────────
        # This model receives both text prompts AND base64-encoded page images.
        # max_tokens=2000 is enough for the JSON response per page.
        self.vision_llm = ChatOpenAI(
            model=VISION_MODEL,
            temperature=0.1,          # Low temp for consistent classification
            max_tokens=2000,          # JSON output per page is ~500 tokens
            openai_api_key=OPENAI_API_KEY,
        )
        # ── Text LLM ───────────────────────────────────────────────────
        # Used only in the consolidation step where we send the accumulated
        # text findings (no images) and ask for a merged section tree.
        # max_tokens=4000 because the consolidated output can be large.
        self.text_llm = ChatOpenAI(
            model=LLM_MODEL,
            temperature=LLM_TEMPERATURE,
            max_tokens=4000,          # Consolidation output can be verbose
            openai_api_key=OPENAI_API_KEY,
        )
        # Storage for results so other parts of the pipeline can access them
        self.discovered_sections: Dict = {}   # Populated by discover_sections()
        self.validation_result: Dict = {}     # Populated by validate_document()

    # ── Helper: Render a PDF page to base64 PNG ──────────────────────────
    @staticmethod
    def _render_page_to_base64(pdf_path: str, page_idx: int,
                               zoom: float = 1.5) -> str:
        """
        Render a single PDF page to a base64-encoded PNG string.
        
        Args:
            pdf_path: Path to the PDF file
            page_idx: 0-based page index (page 1 = index 0)
            zoom: Rendering zoom factor. 1.5 gives 72*1.5 = 108 DPI —
                  enough for the vision model to read text clearly
                  without producing excessively large images.
        
        Returns:
            Base64-encoded PNG string (ready for data:image/png;base64,... URLs)
        
        Implementation:
            1. Open PDF with PyMuPDF (fitz)
            2. Select the page by 0-based index
            3. Create a zoom matrix (scales both X and Y by the zoom factor)
            4. Render page to a pixmap (raster image in memory)
            5. Convert pixmap to PNG bytes
            6. Base64-encode the PNG bytes for API transmission
        """
        doc = fitz.open(pdf_path)
        page = doc[page_idx]
        mat = fitz.Matrix(zoom, zoom)        # Zoom matrix for resolution control
        pix = page.get_pixmap(matrix=mat)    # Rasterize the page into pixels
        png_bytes = pix.tobytes("png")       # Serialize pixels to PNG format
        doc.close()
        return base64.b64encode(png_bytes).decode("utf-8")  # Encode as base64 text

    # ── Helper: Get total page count ─────────────────────────────────────
    @staticmethod
    def _get_page_count(pdf_path: str) -> int:
        """Open the PDF briefly just to count pages, then close it."""
        doc = fitz.open(pdf_path)
        count = len(doc)
        doc.close()
        return count

    # ── Helper: Call vision model with text + image ──────────────────────
    def _call_vision(self, prompt_text: str, image_base64: str) -> str:
        """
        Send a combined text + image prompt to the vision model.
        
        The OpenAI vision API expects a HumanMessage with a list of content
        blocks: one "text" block for the prompt, and one "image_url" block
        with a data URI containing the base64 PNG.
        
        Args:
            prompt_text: The text instructions for the model
            image_base64: Base64-encoded PNG image string
        
        Returns:
            The model's text response (typically JSON)
        """
        # Build a multi-modal message with text + image content blocks
        message = HumanMessage(
            content=[
                # Block 1: Text instructions (the prompt)
                {"type": "text", "text": prompt_text},
                # Block 2: The page image as a data URI
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/png;base64,{image_base64}"
                    },
                },
            ]
        )
        # Send to the vision LLM and extract the text response
        response = self.vision_llm.invoke([message])
        return response.content

    # ── Helper: Parse JSON from LLM response ────────────────────────────
    @staticmethod
    def _parse_json_response(text: str) -> Dict:
        """
        Robustly extract JSON from an LLM response.
        
        LLMs sometimes wrap JSON in markdown code fences like:
            ```json
            {"key": "value"}
            ```
        This method strips those fences before parsing.
        
        Returns:
            Parsed dict, or empty dict {} if parsing fails.
        """
        text = text.strip()
        # Strip markdown JSON code fences if present
        if "```json" in text:
            text = text.split("```json")[1].split("```")[0]
        elif "```" in text:
            text = text.split("```")[1].split("```")[0]
        try:
            return json.loads(text.strip())
        except json.JSONDecodeError:
            return {}  # Graceful fallback — caller should handle empty dict

    # =====================================================================
    # 1. DOCUMENT VALIDATION
    # =====================================================================
    # The scoring algorithm:
    #   - Start at VALIDATION_INITIAL_SCORE (50)
    #   - For each of the first VALIDATION_MAX_PAGES (5) pages:
    #     • Render page to PNG image
    #     • Send to vision model with DOCUMENT_VALIDATION_PROMPT
    #     • If model says is_mf_document=True  → score += 10
    #     • If model says is_mf_document=False → score -= 10
    #   - Final score ≥ VALIDATION_THRESHOLD (65) → VALID
    #   - Final score < 65 → REJECTED (pipeline stops)
    #
    # Why a scoring system instead of just checking page 1?
    #   - Cover pages or legal disclaimers might not look like MF content
    #   - A scoring system is more robust to edge cases
    #   - Multiple pages give higher confidence
    # =====================================================================

    def validate_document(self, pdf_path: str) -> Tuple[bool, int, str]:
        """
        Score-based document validation using vision model.
        
        Args:
            pdf_path: Path to the PDF file to validate
        
        Returns:
            Tuple of:
                is_valid (bool):   True if score ≥ threshold
                final_score (int): The accumulated score
                summary (str):     Human-readable summary of per-page results
        """
        total_pages = self._get_page_count(pdf_path)
        # Don't scan more pages than exist in the document
        pages_to_scan = min(total_pages, VALIDATION_MAX_PAGES)

        score = VALIDATION_INITIAL_SCORE   # Start at 50
        reasons: List[str] = []            # Collect per-page explanations

        print(f"📋 Validating document ({pages_to_scan} pages to scan) …")
        print(f"   Initial score: {score}")

        for page_idx in range(pages_to_scan):
            try:
                # Step 1: Render the page to a base64 PNG image
                image_b64 = self._render_page_to_base64(pdf_path, page_idx)
                
                # Step 2: Send the image to the vision model for classification
                response_text = self._call_vision(
                    DOCUMENT_VALIDATION_PROMPT, image_b64
                )
                
                # Step 3: Parse the JSON response
                result = self._parse_json_response(response_text)

                # Step 4: Update score based on classification
                is_mf = result.get("is_mf_document", False)
                reason = result.get("reason", "No reason provided")

                if is_mf:
                    score += VALIDATION_SCORE_DELTA   # +10 for MF content
                    print(f"   Page {page_idx + 1}: +{VALIDATION_SCORE_DELTA} (MF content) → {score}  | {reason}")
                else:
                    score -= VALIDATION_SCORE_DELTA   # -10 for non-MF content
                    print(f"   Page {page_idx + 1}: −{VALIDATION_SCORE_DELTA} (not MF)    → {score}  | {reason}")

                reasons.append(f"Page {page_idx + 1}: {reason}")

            except Exception as e:
                # On error, skip the page without changing the score
                print(f"   Page {page_idx + 1}: Error — {e}")
                reasons.append(f"Page {page_idx + 1}: Error — {e}")

        # Final verdict: compare accumulated score against threshold
        is_valid = score >= VALIDATION_THRESHOLD
        summary = f"Score: {score}/100. " + " | ".join(reasons[:3])

        # Store the full result for later reference (e.g., by the API)
        self.validation_result = {
            "is_valid": is_valid,
            "score": score,
            "threshold": VALIDATION_THRESHOLD,
            "pages_scanned": pages_to_scan,
            "reasons": reasons,
            "summary": summary,
        }

        tag = "✅ VALID" if is_valid else "❌ REJECTED"
        print(f"\n   Validation: {tag}  (score {score}, threshold {VALIDATION_THRESHOLD})")
        return is_valid, score, summary

    # =====================================================================
    # 2. SECTION DISCOVERY
    # =====================================================================
    # This is the most important method in the pipeline. It replaces the
    # old hardcoded MF_SECTIONS dictionary with a dynamic approach:
    #
    # Phase 1 — Per-page scanning:
    #   For each page in the PDF:
    #     1. Render the page to PNG
    #     2. Build a prompt that includes sliding-window memory context
    #        (what was found on the previous 3 pages)
    #     3. Send the image + prompt to the vision model
    #     4. The model returns section titles, descriptions, keywords,
    #        subsections, and whether each is a continuation
    #     5. Store the AI's response in the sliding window memory
    #
    # Phase 2 — Consolidation:
    #   After all pages are scanned, we have N separate page analyses.
    #   The text LLM merges them into a single, deduplicated section tree:
    #     - Sections spanning multiple pages are merged
    #     - Duplicate subsections are removed
    #     - Keywords from all pages are combined
    #     - The result is ordered as sections appear in the document
    # =====================================================================

    def discover_sections(self, pdf_path: str) -> Dict:
        """
        Dynamically discover all sections in the document.
        
        This is a two-phase process:
            Phase 1: Vision model scans each page with sliding-window context
            Phase 2: Text LLM consolidates page analyses into a section tree
        
        Args:
            pdf_path: Path to the PDF file
        
        Returns:
            dict: {section_key: {title, description, subsections, keywords, pages}}
            Example: {
                "investment_objective": {
                    "title": "Investment Objective",
                    "description": "Fund's primary investment goal",
                    "subsections": ["Primary Objective", "Secondary Objective"],
                    "keywords": ["growth", "income", "capital appreciation"],
                    "pages": [2, 3]
                }
            }
        """
        total_pages = self._get_page_count(pdf_path)
        # Create a fresh sliding window memory for this scan
        memory = SlidingWindowMemory(window_size=PAGE_MEMORY_WINDOW)
        page_analyses: List[Dict] = []  # Accumulates per-page JSON results

        print(f"\n🔍 Discovering sections across {total_pages} pages …")

        # ── Phase 1: Scan each page with vision model ────────────────────
        for page_idx in range(total_pages):
            page_num = page_idx + 1  # 1-based for display
            print(f"   Scanning page {page_num}/{total_pages} …", end=" ")

            try:
                # Get the sliding window context from previous pages.
                # On page 1, this returns "" (no prior context).
                # On page 4, this returns findings from pages 1-3.
                memory_context = memory.get_context_string()
                
                # Fill in the prompt template with memory context and page number
                prompt = PAGE_SECTION_SCAN_PROMPT.format(
                    memory_context=memory_context,
                    page_num=page_num,
                )

                # Render the current page to a PNG image
                image_b64 = self._render_page_to_base64(pdf_path, page_idx)
                
                # Send image + prompt to vision model
                response_text = self._call_vision(prompt, image_b64)
                result = self._parse_json_response(response_text)

                if result:
                    # Store this page's analysis for later consolidation
                    page_analyses.append(result)

                    # Update the sliding window memory with what we found.
                    # This context will be available to the next page's scan.
                    page_summary = result.get("page_summary", f"Page {page_num} analyzed")
                    section_titles = [
                        s.get("title", "") for s in result.get("sections_found", [])
                    ]
                    memory.add_exchange(
                        user_input=f"Scan page {page_num}",
                        ai_output=(
                            f"Page {page_num}: {page_summary}. "
                            f"Sections: {', '.join(section_titles)}"
                        ),
                    )
                    print(f"→ Found {len(section_titles)} sections: {section_titles}")
                else:
                    print("→ No structured response (model returned unparseable output)")

            except Exception as e:
                print(f"→ Error: {e}")

        # ── Phase 2: Consolidate all page analyses into section tree ─────
        print("\n   Consolidating sections with text LLM …")
        self.discovered_sections = self._consolidate_sections(page_analyses)

        # Print the discovered structure
        print(f"\n📂 Discovered {len(self.discovered_sections)} sections:")
        for key, sec in self.discovered_sections.items():
            n_sub = len(sec.get("subsections", []))
            n_kw = len(sec.get("keywords", []))
            print(f"   • {sec['title']}  ({n_sub} subsections, {n_kw} keywords)")

        return self.discovered_sections

    # ── Consolidation: merge page analyses into single tree ──────────────
    def _consolidate_sections(self, page_analyses: List[Dict]) -> Dict:
        """
        Use text LLM to merge page-by-page findings into a clean section tree.
        
        This solves the problem of sections spanning multiple pages:
        e.g., "Risk Factors" found on pages 3, 4, and 5 should become
        ONE section entry with pages=[3, 4, 5].
        
        Args:
            page_analyses: List of per-page JSON dicts from the vision scanner
        
        Returns:
            dict: Consolidated {section_key: section_config}
        """
        # Build a readable text summary of ALL page analyses
        # This text gets inserted into SECTION_CONSOLIDATION_PROMPT
        analyses_text = ""
        for analysis in page_analyses:
            page_num = analysis.get("page_number", "?")
            analyses_text += f"\n--- Page {page_num} ---\n"
            analyses_text += f"Summary: {analysis.get('page_summary', 'N/A')}\n"
            for sec in analysis.get("sections_found", []):
                analyses_text += f"  Section: {sec.get('title', 'Unknown')}\n"
                analyses_text += f"    Description: {sec.get('description', '')}\n"
                kw = sec.get("keywords", [])
                analyses_text += f"    Keywords: {', '.join(kw)}\n"
                subs = sec.get("subsections", [])
                if subs:
                    analyses_text += f"    Subsections: {', '.join(subs)}\n"
                analyses_text += f"    Continuation: {sec.get('is_continuation', False)}\n"
                analyses_text += f"    Content: {sec.get('content_summary', '')}\n"

        # Send the assembled text to the text LLM for consolidation
        prompt = SECTION_CONSOLIDATION_PROMPT.format(page_analyses=analyses_text)
        response = self.text_llm.invoke(prompt)
        result = self._parse_json_response(response.content)

        # Convert the LLM's JSON list into a keyed dictionary for easy lookup.
        # The LLM returns: {"sections": [{section_key, title, ...}, ...]}
        # We transform it to: {section_key: {title, description, ...}}
        sections_dict: Dict = {}
        for idx, sec in enumerate(result.get("sections", [])):
            # Use the LLM-generated key, or fall back to "section_0", "section_1", etc.
            key = sec.get("section_key", f"section_{idx}")
            sections_dict[key] = {
                "title": sec.get("title", f"Section {idx + 1}"),
                "description": sec.get("description", ""),
                "subsections": sec.get("subsections", []),
                "keywords": sec.get("keywords", []),
                "pages": sec.get("pages", []),
            }

        return sections_dict


# ── Verify the class is ready ────────────────────────────────────────────
print("✅ DocumentScanner class defined")
print("   • validate_document()  — vision-based MF detection with scoring system")
print("   • discover_sections()  — page-by-page vision scanning + LLM consolidation")
print("   • Replaces the old hardcoded MF_SECTIONS with dynamic discovery")

## 3 · Document Processor — Docling PDF Parsing & Chunking

Converts the PDF into structured chunks using **Docling**:

- **`DocumentConverter`** — Reads the PDF, maintains reading order, detects element types (text, tables, headings)
- **`HybridChunker`** — Splits into semantic chunks (max 512 tokens) with metadata
- Each chunk carries: `page_number`, `bbox` (bounding box), `element_type`, `headings`

The bounding boxes are critical for **highlighting chunks in the PDF viewer**.

In [ ]:
# =============================================================================
# CHUNK DATA CLASS — The fundamental data unit that flows through the pipeline
# =============================================================================
# Every piece of text extracted from the PDF is wrapped in this dataclass.
# It carries the text AND its metadata (page, bounding box, type, headings).
#
# This dataclass is the bridge between:
#   - Docling (PDF parsing)     → creates ChunkWithMetadata objects
#   - ChromaDB (vector store)   → indexes the text + stores metadata
#   - RAG retrieval             → returns chunks with their metadata
#   - UI (Streamlit)            → uses bbox for highlighting, page for navigation
#
# Data flow:
#   PDF → Docling → ChunkWithMetadata → ChromaDB → RAG retrieval → LLM
#                                                          ↓
#                                                    UI Highlighting
# =============================================================================

from typing import Optional, Any     # Optional for nullable types, Any for generic
from dataclasses import dataclass    # Lightweight class for structured data

from docling.document_converter import DocumentConverter  # Core PDF parser
from docling.chunking import HybridChunker                # Semantic text splitter
from docling.datamodel.document import TableItem          # Type for detected tables


@dataclass
class ChunkWithMetadata:
    """
    A single document chunk with all its metadata for RAG and UI display.
    
    This is the fundamental data unit that flows through the pipeline:
        PDF → Docling → ChunkWithMetadata → ChromaDB → RAG retrieval → LLM
    
    Attributes:
        chunk_id (int):              Unique index within the document (0, 1, 2, …)
        text (str):                  The actual text content of this chunk
        page_number (int):           1-based page number (for PDF viewer navigation)
        bbox (Dict[str, float]):     Bounding box as {left, top, right, bottom} in PDF
                                     coordinate space — used to draw highlight rectangles
                                     in the Streamlit PDF viewer
        element_type (str):          Docling element label: 'text', 'table', 'heading',
                                     'list_item', 'caption', 'page_header', etc.
        headings (List[str]):        Hierarchical heading trail that this chunk belongs to.
                                     E.g., ["Risk Factors", "Market Risk"] means this chunk
                                     is under Risk Factors > Market Risk
    """
    chunk_id: int
    text: str
    page_number: int
    bbox: Optional[Dict[str, float]]   # None if Docling couldn't determine bbox
    element_type: str
    headings: List[str]
    
    def to_dict(self) -> Dict:
        """Convert to a plain dictionary for JSON serialisation (API responses)."""
        return {
            "chunk_id": self.chunk_id,
            "text": self.text,
            "page_number": self.page_number,
            "bbox": self.bbox,
            "element_type": self.element_type,
            "headings": self.headings
        }


print("✅ ChunkWithMetadata dataclass defined")
print("   Fields: chunk_id, text, page_number, bbox, element_type, headings")
print("   to_dict() → plain dict for JSON API responses")

In [ ]:
# =============================================================================
# DOCUMENT PROCESSOR — Docling-based PDF extraction & chunking
# =============================================================================
# This class handles the "traditional" PDF parsing pipeline using Docling:
#
#   1. DocumentConverter reads the PDF and produces a structured document
#      object with proper reading order, element types (heading, text, table),
#      and bounding boxes for each element.
#
#   2. HybridChunker splits the structured document into semantic chunks of
#      max 512 tokens each. "Hybrid" means it combines:
#      - Rule-based splitting (respecting headings, paragraphs, tables)
#      - Token-count-based splitting (ensuring no chunk exceeds max_tokens)
#      - Peer merging (merging small adjacent elements into one chunk)
#
# Why Docling instead of simple text extraction (PyPDF2/pdfplumber)?
#   - ML-based layout analysis → proper reading order
#   - Table structure recognition → tables as DataFrames
#   - Heading hierarchy → section trail for each chunk
#   - Bounding boxes → PDF highlighting in the UI
#
# Note: Docling's DocumentConverter is heavy — it loads ML models (~2GB)
# for layout analysis and table detection. We lazy-load it so the startup
# cost is paid only when actually processing a document.
# =============================================================================


class DocumentProcessor:
    """
    Processes PDF documents using Docling to extract structured content
    with proper reading order, element types, and bounding boxes.
    
    Usage:
        processor = DocumentProcessor()
        chunks = processor.process_document("path/to/fund.pdf")
        # chunks is a list of ChunkWithMetadata objects ready for RAG indexing
    
    Key design choices:
        - Lazy initialization: Docling's converter loads ML models (~2GB),
          so we only initialize it when process_document() is first called.
        - merge_peers=True in HybridChunker: Adjacent small elements (e.g.,
          two short paragraphs) are merged into one chunk for better context.
    """
    
    def __init__(self, tokenizer_model: str = TOKENIZER_MODEL,
                 max_tokens: int = MAX_TOKENS_PER_CHUNK):
        """
        Args:
            tokenizer_model: HuggingFace tokenizer name for token counting.
                             Default: "sentence-transformers/all-MiniLM-L6-v2"
            max_tokens:      Maximum tokens per chunk. Default: 512.
                             Larger → more context per chunk, but less precise retrieval.
                             Smaller → more precise, but might split important info.
        """
        self.tokenizer_model = tokenizer_model
        self.max_tokens = max_tokens
        self.converter = None     # Lazy-loaded (heavy: loads layout ML models)
        self.chunker = None       # Lazy-loaded
        self.document = None      # Stores the parsed Docling Document object
        self.chunks: List[ChunkWithMetadata] = []  # All extracted chunks
        
    def _initialize_converter(self):
        """
        Lazy-load Docling's DocumentConverter.
        This is the most expensive initialization step — it downloads and loads
        ML models for layout analysis, OCR, and table structure recognition.
        Only called once; subsequent calls are no-ops.
        """
        if self.converter is None:
            print("   Initializing Docling DocumentConverter …")
            self.converter = DocumentConverter()
            print("   DocumentConverter ready!")
            
    def _initialize_chunker(self):
        """
        Lazy-load the HybridChunker.
        
        Parameters:
            tokenizer: Which tokenizer to use for counting tokens per chunk.
            max_tokens: Maximum tokens before splitting. 512 ≈ 350-400 words.
            merge_peers: If True, adjacent small elements (e.g., two short
                         paragraphs under the same heading) are merged into
                         a single chunk. This gives the RAG retriever more
                         context per chunk.
        """
        if self.chunker is None:
            print("   Initializing HybridChunker …")
            self.chunker = HybridChunker(
                tokenizer=self.tokenizer_model,
                max_tokens=self.max_tokens,
                merge_peers=True    # Merge small adjacent elements for better context
            )
            print("   HybridChunker ready!")
    
    def process_document(self, pdf_path: str) -> List[ChunkWithMetadata]:
        """
        Full PDF processing pipeline:
            1. Initialize Docling components (lazy — only on first call)
            2. Convert PDF → Docling structured document
               (handles OCR, table detection, reading order, layout analysis)
            3. Chunk document → list of semantic chunks with metadata
        
        Args:
            pdf_path: Path to the PDF file to process
        
        Returns:
            List of ChunkWithMetadata objects, each containing text, page number,
            bounding box (for highlighting), element type, and heading trail.
        """
        # Lazy-load the heavy components
        self._initialize_converter()
        self._initialize_chunker()
        
        # ── Step 1: Convert the PDF using Docling ────────────────────────
        # Docling's converter handles: OCR, layout analysis, table detection,
        # reading order determination, heading hierarchy extraction.
        # The result.document is a rich structured object we can iterate over.
        print(f"   Converting document: {pdf_path}")
        result = self.converter.convert(pdf_path)
        self.document = result.document
        print(f"   Document converted: {self.document.num_pages} pages")
        
        # ── Step 2: Generate semantic chunks ─────────────────────────────
        # The HybridChunker walks through the structured document and splits
        # it into chunks that respect heading boundaries and table structures.
        print("   Generating semantic chunks …")
        raw_chunks = list(self.chunker.chunk(self.document))
        print(f"   Generated {len(raw_chunks)} raw chunks")
        
        # ── Step 3: Extract metadata for each chunk ──────────────────────
        # Each raw chunk from Docling carries metadata in its .meta attribute.
        # We extract: page_number, bounding_box, element_type, headings.
        self.chunks = []
        for idx, chunk in enumerate(raw_chunks):
            chunk_with_meta = self._extract_chunk_metadata(idx, chunk)
            self.chunks.append(chunk_with_meta)
        
        print(f"   Processed {len(self.chunks)} chunks with metadata")
        return self.chunks
    
    def _extract_chunk_metadata(self, idx: int, chunk) -> ChunkWithMetadata:
        """
        Extract page number, bounding box, element type, and headings
        from a raw Docling chunk object.
        
        Docling chunk structure:
            chunk.text           → the actual text
            chunk.meta.headings  → list of section headings this chunk belongs to
            chunk.meta.doc_items → list of document elements in this chunk
              doc_item.label     → element type (text, table, heading, etc.)
              doc_item.prov      → provenance: page_no, bbox (bounding box)
        
        We only use the FIRST doc_item's provenance (first element in the chunk)
        for page number and bbox. This is a simplification — a chunk could span
        elements from different pages, but in practice it rarely does with
        max_tokens=512.
        
        Args:
            idx: 0-based chunk index (becomes chunk_id)
            chunk: Raw Docling chunk object
        
        Returns:
            ChunkWithMetadata with all fields populated
        """
        # Get the text content (handle edge cases where .text might not exist)
        text = chunk.text if hasattr(chunk, 'text') else str(chunk)
        
        # Default values (used if metadata extraction fails)
        page_number = 1         # Fallback to page 1
        bbox = None             # No bounding box available
        element_type = "text"   # Default element type
        headings = []           # No heading trail
        
        # ── Extract metadata from the chunk's .meta attribute ────────────
        if hasattr(chunk, 'meta') and chunk.meta:
            meta = chunk.meta
            
            # 1. Section headings this chunk belongs to
            #    e.g., ["Investment Objective", "Primary Objective"]
            if hasattr(meta, 'headings') and meta.headings:
                headings = meta.headings
            
            # 2. Dig into doc_items for element type and provenance
            #    doc_items is a list of document elements that make up this chunk.
            #    Each doc_item has:
            #      - label: the element type (DocItemLabel enum)
            #      - prov: list of provenance records (page_no, bbox)
            if hasattr(meta, 'doc_items') and meta.doc_items:
                for item in meta.doc_items:
                    # Get element type (e.g., 'text', 'table', 'section_header')
                    if hasattr(item, 'label'):
                        label = item.label
                        element_type = str(label.value) if hasattr(label, 'value') else str(label)
                    
                    # Get page number and bounding box from provenance
                    if hasattr(item, 'prov') and item.prov:
                        for prov in item.prov:
                            if hasattr(prov, 'page_no'):
                                page_number = prov.page_no
                            if hasattr(prov, 'bbox'):
                                b = prov.bbox
                                # Convert Docling bbox to our dict format
                                # l=left, t=top, r=right, b=bottom
                                bbox = {
                                    'left': float(b.l),
                                    'top': float(b.t),
                                    'right': float(b.r),
                                    'bottom': float(b.b)
                                }
                            break  # Use first provenance record only
                    break  # Use first doc_item only (simplification)
        
        return ChunkWithMetadata(
            chunk_id=idx,
            text=text,
            page_number=page_number,
            bbox=bbox,
            element_type=element_type,
            headings=headings
        )
    
    def get_chunks_by_page(self, page_number: int) -> List[ChunkWithMetadata]:
        """
        Get all chunks from a specific page.
        Useful for viewing all content on a given page in the UI.
        """
        return [c for c in self.chunks if c.page_number == page_number]
    
    def get_tables(self) -> List[Dict]:
        """
        Extract all tables detected by Docling as Pandas DataFrames.
        
        Docling's layout analysis model detects tables in the PDF.
        Each TableItem can be exported to a pandas DataFrame for display.
        
        Returns:
            List of dicts, each containing:
                - 'dataframe': pandas DataFrame of the table
                - 'page_number': which page the table is on
                - 'bbox': bounding box coordinates for highlighting
        """
        tables = []
        if self.document is None:
            return tables
        
        # iterate_items() walks through all document elements with their
        # nesting level. We filter for TableItem instances.
        for idx, (item, level) in enumerate(self.document.iterate_items()):
            if isinstance(item, TableItem):
                try:
                    # Export the detected table structure to a pandas DataFrame
                    df = item.export_to_dataframe()
                    
                    # Extract page number and bbox from provenance
                    page_no = 1
                    bbox = None
                    if hasattr(item, 'prov') and item.prov:
                        for prov in item.prov:
                            if hasattr(prov, 'page_no'):
                                page_no = prov.page_no
                            if hasattr(prov, 'bbox'):
                                b = prov.bbox
                                bbox = {
                                    'left': float(b.l),
                                    'top': float(b.t),
                                    'right': float(b.r),
                                    'bottom': float(b.b)
                                }
                            break
                    tables.append({
                        'dataframe': df,
                        'page_number': page_no,
                        'bbox': bbox
                    })
                except Exception as e:
                    print(f"   Could not extract table: {e}")
        return tables


# ── Verify the class is ready ────────────────────────────────────────────
print("✅ DocumentProcessor class defined")
print("   • process_document()    — PDF → Docling → semantic chunks with bbox metadata")
print("   • get_chunks_by_page()  — filter chunks by page number")
print("   • get_tables()          — extract detected tables as DataFrames")

## 4 · RAG Engine — ChromaDB + Re-ranking + Chat

The RAG Engine handles the entire retrieval-augmented generation pipeline:

1. **`index_chunks()`** — Embeds chunks using `text-embedding-3-small` and stores them in ChromaDB
2. **`retrieve()`** — Similarity search against the vector store (top 10)
3. **`rerank()`** — LLM-based re-ranking: scores each chunk 0–10, keeps top 5
4. **`retrieve_and_rerank()`** — Combines retrieve + rerank in one call
5. **`chat()`** — Full Q&A: retrieve → rerank → GPT-4o-mini answer with context
6. **`search_for_section()`** — Section-specific retrieval with keyword boosting

In [ ]:
# =============================================================================
# RAG ENGINE — ChromaDB vector store + LLM re-ranking + chatbot
# =============================================================================
# This class implements the full Retrieval-Augmented Generation (RAG) pipeline:
#
#   Index Phase:
#     1. Take all ChunkWithMetadata objects from DocumentProcessor
#     2. Embed each chunk's text using OpenAI's text-embedding-3-small model
#     3. Store embeddings + metadata in ChromaDB (persisted to disk)
#
#   Retrieval Phase (two-stage):
#     Stage 1 — Similarity Search:
#       - User query → embed → cosine similarity against ChromaDB → top K chunks
#       - Fast but can return false positives (semantically similar but irrelevant)
#
#     Stage 2 — LLM Re-ranking:
#       - Send the top K chunks to GPT-4o-mini
#       - Model scores each chunk 0-10 for relevance to the query
#       - Keep only the top N highest-scoring chunks
#       - This dramatically improves retrieval precision
#
#   Generation Phase:
#     - Build a prompt with the re-ranked chunks as context
#     - GPT-4o-mini generates an answer grounded in the document
#     - Returns both the answer AND the source chunks (for traceability)
#
# The engine also provides search_for_section() which adds keyword boosting
# from dynamically discovered section keywords for better section retrieval.
# =============================================================================

import chromadb                                # ChromaDB vector database client
from chromadb.config import Settings           # ChromaDB configuration

from langchain_openai import OpenAIEmbeddings, ChatOpenAI  # OpenAI integrations
from langchain_core.documents import Document               # LangChain Document type
from langchain_chroma import Chroma                         # LangChain-ChromaDB bridge


class RAGEngine:
    """
    RAG Engine: Embed → Store → Retrieve → Re-rank → Generate.
    
    This is the backbone of the Q&A and section extraction features.
    Every query goes through: embed query → similarity search → LLM re-rank → answer.
    
    Key components:
        - self.embeddings:      OpenAI embedding model (text → 1536-dim vector)
        - self.llm:             GPT-4o-mini for re-ranking, chat, extraction
        - self.vector_store:    ChromaDB via LangChain (stores document vectors)
        - self.chunks_metadata: In-memory lookup of chunk_id → ChunkWithMetadata
    """
    
    def __init__(self, persist_directory: str = CHROMA_PERSIST_DIR):
        """
        Args:
            persist_directory: Where ChromaDB stores its data on disk.
                               Default: "./chroma_db". Data persists across
                               kernel restarts, avoiding re-embedding costs.
        """
        self.persist_directory = persist_directory
        
        # ── OpenAI Embeddings ────────────────────────────────────────────
        # Converts text into 1536-dimensional vectors for similarity search.
        # text-embedding-3-small: $0.02/1M tokens, good quality/cost ratio.
        print("   Initializing OpenAI Embeddings …")
        self.embeddings = OpenAIEmbeddings(
            model=EMBEDDING_MODEL,
            openai_api_key=OPENAI_API_KEY
        )
        
        # ── Chat LLM ────────────────────────────────────────────────────
        # Used for three tasks:
        #   1. Re-ranking chunks (scoring 0-10 per chunk)
        #   2. Generating chat answers from context
        #   3. Section extraction / summarisation
        print("   Initializing ChatOpenAI …")
        self.llm = ChatOpenAI(
            model=LLM_MODEL,
            temperature=LLM_TEMPERATURE,  # 0.1 for deterministic scoring/answers
            openai_api_key=OPENAI_API_KEY
        )
        
        # ── Vector Store ─────────────────────────────────────────────────
        # ChromaDB instance, created when index_chunks() is called.
        # None until then (calling retrieve() before indexing raises an error).
        self.vector_store = None
        
        # ── In-memory chunk metadata lookup ──────────────────────────────
        # Maps chunk_id → ChunkWithMetadata for quick access.
        # This is separate from ChromaDB because ChromaDB stores strings,
        # not our custom dataclass objects.
        self.chunks_metadata: Dict[int, ChunkWithMetadata] = {}
        
        print("   RAG Engine initialized!")
    
    # =====================================================================
    # INDEX — Embed and store chunks in ChromaDB
    # =====================================================================
    def index_chunks(self, chunks: List[ChunkWithMetadata],
                     collection_name: str = COLLECTION_NAME):
        """
        Embed and store all document chunks in ChromaDB.
        
        Process:
            1. Convert each ChunkWithMetadata → LangChain Document
               (LangChain Document = page_content string + metadata dict)
            2. LangChain-Chroma embeds all documents using OpenAI embeddings
            3. Stores vectors + metadata in ChromaDB on disk
        
        Args:
            chunks: List of ChunkWithMetadata from DocumentProcessor
            collection_name: ChromaDB collection name (default: "mf_prospectus")
                             Use different names to keep multiple documents separate.
        
        Returns:
            The Chroma vector store object
        """
        print(f"   Indexing {len(chunks)} chunks into ChromaDB …")
        
        # Keep a local lookup table for chunk metadata (by ID)
        self.chunks_metadata = {chunk.chunk_id: chunk for chunk in chunks}
        
        # Convert our ChunkWithMetadata objects to LangChain Document objects.
        # LangChain's Document has: page_content (string) + metadata (dict).
        # We store chunk_id, page_number, element_type, headings, and bbox
        # as metadata so we can retrieve them after similarity search.
        documents = []
        for chunk in chunks:
            doc = Document(
                page_content=chunk.text,   # The text that gets embedded
                metadata={
                    "chunk_id": chunk.chunk_id,          # For cross-referencing
                    "page_number": chunk.page_number,    # For page navigation in UI
                    "element_type": chunk.element_type,  # For filtering (text/table/heading)
                    "headings": json.dumps(chunk.headings),  # JSON string (ChromaDB needs strings)
                    "bbox": json.dumps(chunk.bbox) if chunk.bbox else None  # For PDF highlighting
                }
            )
            documents.append(doc)
        
        # Create the vector store. Chroma.from_documents() does:
        #   1. Embeds all page_content strings using self.embeddings
        #   2. Stores vectors + metadata in ChromaDB
        #   3. Persists to disk at persist_directory
        self.vector_store = Chroma.from_documents(
            documents=documents,
            embedding=self.embeddings,
            persist_directory=self.persist_directory,
            collection_name=collection_name
        )
        
        print(f"   ✅ Indexed {len(documents)} chunks!")
        return self.vector_store
    
    # =====================================================================
    # RETRIEVE — Similarity search against ChromaDB
    # =====================================================================
    def retrieve(self, query: str, top_k: int = RERANK_TOP_K) -> List[Document]:
        """
        Stage 1 of retrieval: Vector similarity search.
        
        The query text is embedded using the same model, then ChromaDB
        finds the top_k most similar vectors by cosine distance.
        
        Args:
            query: Natural language search query
            top_k: Number of results to return (default: 10)
        
        Returns:
            List of LangChain Document objects with page_content + metadata
        
        Raises:
            ValueError: If called before index_chunks()
        """
        if self.vector_store is None:
            raise ValueError("No vector store. Call index_chunks() first.")
        results = self.vector_store.similarity_search(query, k=top_k)
        return results
    
    # =====================================================================
    # RE-RANK — LLM-based chunk scoring
    # =====================================================================
    def rerank(self, query: str, documents: List[Document]) -> List[Tuple[Document, float]]:
        """
        Stage 2 of retrieval: LLM-based re-ranking.
        
        Why re-rank?
            Vector similarity is fast but imprecise. "Investment objective"
            might be similar to "investment horizon" in embedding space,
            but they answer different questions. The LLM can understand
            semantic nuance and score relevance more accurately.
        
        Process:
            1. Format all chunks into numbered text blocks (max 500 chars each)
            2. Send to GPT-4o-mini with RERANK_PROMPT
            3. Model returns JSON: [{"chunk_id": 0, "score": 8}, ...]
            4. Sort by score descending
        
        Args:
            query: The search query
            documents: Candidate documents from similarity search
        
        Returns:
            List of (Document, score) tuples sorted by score descending.
            On error, falls back to score=5.0 for all documents.
        """
        if not documents:
            return []
        
        # Build the chunks text block for the re-rank prompt.
        # We truncate each chunk to 500 chars to keep the prompt within
        # token limits while still giving the LLM enough context to judge.
        chunks_text = ""
        for idx, doc in enumerate(documents):
            chunks_text += f"\n[Chunk {idx}]:\n{doc.page_content[:500]}...\n"
        
        # Fill in the RERANK_PROMPT template with query and chunks
        prompt = RERANK_PROMPT.format(query=query, chunks=chunks_text)
        
        try:
            # Call the LLM to score each chunk
            response = self.llm.invoke(prompt)
            scores_text = response.content.strip()
            
            # Parse the JSON response (strip markdown fences if present)
            if "```json" in scores_text:
                scores_text = scores_text.split("```json")[1].split("```")[0]
            elif "```" in scores_text:
                scores_text = scores_text.split("```")[1].split("```")[0]
            
            scores = json.loads(scores_text)
            
            # Map the LLM's scores back to the original Document objects.
            # The LLM returns chunk_id (0-based index into our chunks list).
            scored_docs = []
            for score_item in scores:
                chunk_id = score_item.get('chunk_id', 0)
                score = score_item.get('score', 0)
                if 0 <= chunk_id < len(documents):
                    scored_docs.append((documents[chunk_id], score))
            
            # Sort by score descending — highest relevance first
            scored_docs.sort(key=lambda x: x[1], reverse=True)
            return scored_docs
            
        except Exception as e:
            # If re-ranking fails (e.g., invalid JSON from LLM),
            # fall back to returning all documents with a neutral score of 5.
            print(f"   Re-ranking failed: {e}")
            return [(doc, 5.0) for doc in documents]
    
    # =====================================================================
    # RETRIEVE + RE-RANK — Full two-stage pipeline
    # =====================================================================
    def retrieve_and_rerank(self, query: str, 
                           initial_k: int = RERANK_TOP_K,
                           final_k: int = FINAL_TOP_K) -> List[Dict]:
        """
        Complete two-stage retrieval pipeline in one call.
        
        Pipeline:
            1. Retrieve initial_k chunks by vector similarity (fast, broad)
            2. LLM re-ranks all initial_k chunks with scores 0-10
            3. Keep only the top final_k chunks (accurate, filtered)
        
        Args:
            query: Natural language search query
            initial_k: How many chunks to retrieve in stage 1 (default: 10)
            final_k: How many to keep after re-ranking (default: 5)
        
        Returns:
            List of dicts, each containing:
                text, score, chunk_id, page_number, element_type, headings, bbox
        """
        # Stage 1: Broad similarity search
        documents = self.retrieve(query, top_k=initial_k)
        
        # Stage 2: Precise LLM re-ranking
        reranked = self.rerank(query, documents)
        
        # Package the top results as dicts with all metadata for downstream use.
        # We deserialise the JSON-encoded metadata fields (headings, bbox)
        # back into Python objects.
        results = []
        for doc, score in reranked[:final_k]:
            metadata = doc.metadata
            # ChromaDB stores bbox and headings as JSON strings — parse them back
            bbox = json.loads(metadata.get('bbox')) if metadata.get('bbox') else None
            headings = json.loads(metadata.get('headings', '[]'))
            
            results.append({
                'text': doc.page_content,        # The chunk's text
                'score': score,                  # Re-rank score (0-10)
                'chunk_id': metadata.get('chunk_id'),   # For cross-referencing
                'page_number': metadata.get('page_number'),  # For UI navigation
                'element_type': metadata.get('element_type'),  # text/table/heading
                'headings': headings,            # Section heading trail
                'bbox': bbox                     # Bounding box for highlighting
            })
        
        return results
    
    # =====================================================================
    # CHAT — Q&A with the document
    # =====================================================================
    def chat(self, query: str,
             context_chunks: Optional[List[Dict]] = None) -> Tuple[str, List[Dict]]:
        """
        Chatbot Q&A: Ask a question and get an answer grounded in the document.
        
        Flow:
            1. If context_chunks not provided, run retrieve_and_rerank()
            2. Format the best chunks into a text block
            3. Insert into CHAT_SYSTEM_PROMPT with the user's query
            4. GPT-4o-mini generates an answer from the context
        
        Args:
            query: The user's natural language question
            context_chunks: Pre-retrieved chunks (optional — if you already
                            have them from a previous search). If None,
                            the method does its own retrieve+rerank.
        
        Returns:
            Tuple of:
                answer (str): The generated answer text
                context_chunks (List[Dict]): The source chunks used (for traceability)
        """
        # Step 1: Get relevant context chunks (retrieve + rerank if not provided)
        if context_chunks is None:
            context_chunks = self.retrieve_and_rerank(query)
        
        # Step 2: Build a formatted context string from the retrieved chunks.
        # Each chunk is labelled with its index and page number so the LLM
        # can reference specific pages in its answer.
        context = ""
        for idx, chunk in enumerate(context_chunks):
            page = chunk.get('page_number', 'N/A')
            context += f"\n[Chunk {idx+1} - Page {page}]:\n{chunk['text']}\n"
        
        # Step 3: Fill in the CHAT_SYSTEM_PROMPT and call the LLM
        prompt = CHAT_SYSTEM_PROMPT.format(context=context, query=query)
        response = self.llm.invoke(prompt)
        
        # Return answer + sources so the user can verify where info came from
        return response.content, context_chunks
    
    # =====================================================================
    # SECTION SEARCH — Keyword-boosted retrieval for section extraction
    # =====================================================================
    def search_for_section(self, section_title: str, section_description: str,
                          subsections: List[str],
                          keywords: Optional[List[str]] = None) -> List[Dict]:
        """
        Retrieve chunks relevant to a specific document section.
        
        This is used by SectionExtractor to find all chunks related to
        a discovered section (e.g., "Investment Objective").
        
        Key difference from regular retrieve_and_rerank():
            - Builds a more comprehensive query from title + description + subsections
            - Appends dynamically discovered keywords for better recall
              (e.g., "NAV", "AUM", "benchmark" for a fund overview section)
            - Uses higher initial_k (15) for broader initial coverage
            - Returns more results (final_k=10) because sections need more context
        
        Args:
            section_title: e.g., "Investment Objective"
            section_description: e.g., "The fund's primary investment goal"
            subsections: e.g., ["Primary Objective", "Investment Strategy"]
            keywords: Discovered keywords (from vision scanning) to boost recall
        
        Returns:
            List of dicts (same format as retrieve_and_rerank)
        """
        # Build a comprehensive search query combining all available info.
        # The more context in the query, the better the embedding similarity.
        query = f"{section_title}: {section_description}. "
        query += f"Looking for: {', '.join(subsections)}"
        
        # Append dynamically discovered keywords for better recall.
        # These keywords came from the vision model's page-level scanning.
        if keywords:
            query += f". Key terms: {', '.join(keywords)}"
        
        # Use wider retrieval (15→10) compared to regular chat (10→5)
        # because sections typically need more comprehensive coverage.
        return self.retrieve_and_rerank(query, initial_k=15, final_k=10)
    
    # =====================================================================
    # ACCESSORS — Get specific chunks by ID or page
    # =====================================================================
    def get_chunk_by_id(self, chunk_id: int) -> Optional[ChunkWithMetadata]:
        """
        Get a specific chunk by its ID from the in-memory lookup.
        Returns None if the chunk_id doesn't exist.
        """
        return self.chunks_metadata.get(chunk_id)
    
    def get_all_chunks_for_page(self, page_number: int) -> List[ChunkWithMetadata]:
        """
        Get all chunks that came from a specific page.
        Useful for showing all content on a page in the PDF viewer.
        """
        return [
            chunk for chunk in self.chunks_metadata.values() 
            if chunk.page_number == page_number
        ]


# ── Verify the class is ready ────────────────────────────────────────────
print("✅ RAGEngine class defined")
print("   • index_chunks()          — embed & store chunks in ChromaDB")
print("   • retrieve()              — stage 1: vector similarity search (top-K)")
print("   • rerank()                — stage 2: LLM scoring 0-10 per chunk")
print("   • retrieve_and_rerank()   — combined: retrieve → rerank → top results")
print("   • chat()                  — Q&A: retrieve → rerank → generate answer")
print("   • search_for_section()    — section retrieval with keyword boosting")

## 5 · Section Extractor — Dynamic Section Extraction via RAG + LLM

Takes the **dynamically discovered sections** (from DocumentScanner) and uses RAG to extract content for each one:

1. For each section: query = title + description + subsections + **keywords** (from discovery)
2. `search_for_section()` retrieves relevant chunks with keyword boosting
3. LLM summarizes the chunks into a section report
4. Per-subsection focused retrieval for granular content

No hardcoded sections — everything works with whatever the scanner discovered.

In [ ]:
# =============================================================================
# SUBSECTION CONTENT — Dataclass for individual subsection data
# =============================================================================
# When we extract a section (e.g., "Risk Factors"), we also do per-subsection
# focused retrieval (e.g., "Market Risk", "Credit Risk"). Each subsection's
# result is stored in this dataclass.
#
# Data flow:
#   SectionExtractor.extract_section() → per-subsection RAG retrieval
#       → SubsectionContent(title="Market Risk", content="...", chunks=[...])
#
# The chunks list provides traceability: for any subsection, you can see
# exactly which document chunks were used to produce the content.
# =============================================================================

from dataclasses import field   # For mutable default values in dataclasses


@dataclass
class SubsectionContent:
    """
    Content extracted for a single subsection within a section.
    
    Example: Under "Risk Factors", a subsection might be "Market Risk"
    with its extracted content and the chunks that supplied the info.
    
    Attributes:
        title (str):       Subsection title (e.g., "Market Risk")
        content (str):     Extracted text content for this subsection
        chunks (List[Dict]): The RAG chunks used to generate this content
                             (for source traceability)
    """
    title: str
    content: str
    chunks: List[Dict] = field(default_factory=list)  # Mutable default → use field()


print("✅ SubsectionContent dataclass defined")
print("   Fields: title, content, chunks")
print("   Used by SectionExtractor for per-subsection focused retrieval results")

In [ ]:
# =============================================================================
# SECTION CONTENT — Dataclass for a complete extracted section
# =============================================================================
# This is the output of SectionExtractor.extract_section(). One SectionContent
# object per discovered section (e.g., "Investment Objective", "Risk Factors").
#
# It contains:
#   - summary: LLM-generated structured summary of the entire section
#   - subsections: List of SubsectionContent with per-sub-topic content
#   - all_chunks: Every chunk used (for traceability / UI source display)
#
# The to_dict() method serialises everything to a plain dict for the
# FastAPI JSON responses.
# =============================================================================


@dataclass
class SectionContent:
    """
    Content extracted for a main section (contains multiple subsections).
    
    This is the output of extract_section() — one per discovered section.
    
    Attributes:
        section_key (str):     Unique identifier (e.g., "investment_objective")
        title (str):           Human-readable title (e.g., "Investment Objective")
        description (str):     What the section covers
        summary (str):         LLM-generated structured summary of the entire section
        subsections (List):    Per-subsection extracted content
        all_chunks (List):     All RAG chunks used for this section
    """
    section_key: str
    title: str
    description: str
    summary: str                                     # LLM-generated structured summary
    subsections: List[SubsectionContent] = field(default_factory=list)
    all_chunks: List[Dict] = field(default_factory=list)  # All source chunks

    def to_dict(self) -> Dict:
        """Serialise to a plain dict for JSON API responses or storage."""
        return {
            "section_key": self.section_key,
            "title": self.title,
            "description": self.description,
            "summary": self.summary,
            "subsections": [
                {"title": sub.title, "content": sub.content, "chunks": sub.chunks}
                for sub in self.subsections
            ],
            "all_chunks": self.all_chunks,
        }


print("✅ SectionContent dataclass defined")
print("   Fields: section_key, title, description, summary, subsections, all_chunks")
print("   to_dict() → plain dict for JSON API responses")

In [ ]:
# =============================================================================
# SECTION EXTRACTOR — Dynamic section extraction using discovered structure
# =============================================================================
# This class bridges the gap between "we know what sections exist" (from
# DocumentScanner.discover_sections) and "here is the actual content for
# each section" (extracted via RAG + LLM summarisation).
#
# For each dynamically discovered section, it:
#   1. Builds a rich search query using the section's title, description,
#      and keywords (all discovered by the vision scanner).
#   2. Calls RAGEngine.search_for_section() with keyword boosting to
#      retrieve the most relevant chunks from ChromaDB.
#   3. Sends those chunks to GPT-4o-mini with SECTION_EXTRACTION_PROMPT
#      to produce a structured summary of the section's content.
#   4. Does additional per-subsection focused retrieval for granular content
#      (e.g., for "Risk Factors" → separate retrieval for "Market Risk",
#      "Credit Risk", "Liquidity Risk" etc.).
#
# The result is a SectionContent object with:
#   - summary: LLM-generated structured summary based on retrieved chunks
#   - subsections: List of SubsectionContent with focused content per sub-topic
#   - all_chunks: All source chunks used (for traceability / UI display)
# =============================================================================


class SectionExtractor:
    """
    Extracts section content using dynamically discovered section definitions + RAG.
    
    This is NOT hardcoded. The constructor receives the `discovered_sections`
    dictionary from DocumentScanner.discover_sections(), which was built by
    the vision model scanning every page of the PDF.
    
    Architecture:
        SectionExtractor
            → uses RAGEngine for chunk retrieval (with keyword boosting)
            → uses ChatOpenAI for LLM summarisation
            → stores results as SectionContent dataclass objects
    
    Usage:
        extractor = SectionExtractor(rag_engine, discovered_sections)
        
        # Extract one section:
        section = extractor.extract_section("investment_objective")
        
        # Extract all sections:
        all_extracted = extractor.extract_all_sections()
    """

    def __init__(self, rag_engine: RAGEngine,
                 discovered_sections: Dict[str, Dict]):
        """
        Args:
            rag_engine: An initialised RAGEngine with indexed chunks.
            discovered_sections: Dict from DocumentScanner.discover_sections().
                Format: {section_key: {title, description, subsections, keywords, pages}}
        """
        self.rag = rag_engine                   # RAG engine for chunk retrieval
        self.sections = discovered_sections     # Dynamic section definitions (NOT hardcoded!)
        
        # Separate LLM instance for section extraction / summarisation.
        # We keep it separate from the RAG engine's LLM for clarity,
        # though they use the same model and settings.
        self.llm = ChatOpenAI(
            model=LLM_MODEL,
            temperature=LLM_TEMPERATURE,
            openai_api_key=OPENAI_API_KEY,
        )
        
        # Stores extracted content keyed by section_key
        self.extracted_sections: Dict[str, SectionContent] = {}

    # =====================================================================
    # EXTRACT ONE SECTION
    # =====================================================================
    def extract_section(self, section_key: str) -> SectionContent:
        """
        Extract content for one specific section.
        
        Process:
            1. Look up the section's config (title, description, subsections, keywords)
            2. RAG retrieval with keyword-boosted query (initial_k=15, final_k=10)
               - Keywords from vision discovery improve retrieval recall
            3. LLM summarisation: send all retrieved chunks to GPT-4o-mini
               with SECTION_EXTRACTION_PROMPT → structured summary
            4. Per-subsection focused retrieval: for each subsection (e.g., "Market Risk"),
               do a targeted retrieve+rerank (initial_k=5, final_k=3)
               to get the most relevant chunks for that specific sub-topic
        
        Args:
            section_key: The section identifier (e.g., "investment_objective")
        
        Returns:
            SectionContent with summary, subsection contents, and source chunks
        
        Raises:
            ValueError: If section_key not found in discovered sections
        """
        if section_key not in self.sections:
            raise ValueError(f"Unknown section: {section_key}")

        # Look up the section's configuration from the discovery results
        cfg = self.sections[section_key]
        title       = cfg["title"]              # e.g., "Investment Objective"
        description = cfg.get("description", "")  # e.g., "The fund's primary goal"
        subsections = cfg.get("subsections", [])  # e.g., ["Primary Objective", "Strategy"]
        keywords    = cfg.get("keywords", [])     # e.g., ["growth", "income", "NAV"]

        print(f"   Extracting: {title}")

        # ── Step 1: RAG retrieval with keyword boosting ──────────────────
        # search_for_section() builds a rich query from title + description +
        # subsections + keywords, then does retrieve (15) + rerank (10).
        # The keywords (from vision discovery) help find relevant chunks that
        # might not match the section title directly.
        chunks = self.rag.search_for_section(
            title, description, subsections, keywords=keywords
        )

        # ── Step 2: Build context text for LLM summarisation ────────────
        # Format all retrieved chunks into a numbered text block
        chunks_text = ""
        for idx, chunk in enumerate(chunks):
            page = chunk.get("page_number", "N/A")
            chunks_text += f"\n[Chunk {idx + 1} - Page {page}]:\n{chunk['text']}\n"

        # ── Step 3: LLM summarisation ────────────────────────────────────
        # Send the chunks to GPT-4o-mini with SECTION_EXTRACTION_PROMPT.
        # The model organises the information by subsection and extracts
        # key figures, data points, and important notes.
        prompt = SECTION_EXTRACTION_PROMPT.format(
            section_title=title,
            section_description=description,
            subsections=", ".join(subsections),
            chunks=chunks_text,
        )
        response = self.llm.invoke(prompt)
        summary = response.content  # The structured summary text

        # ── Step 4: Per-subsection focused retrieval ─────────────────────
        # For each subsection, we do a separate smaller retrieval to get
        # the most relevant chunks specifically for that sub-topic.
        # This provides granular content that the user can inspect per-subsection.
        subsection_contents: List[SubsectionContent] = []
        for sub_title in subsections:
            # Build a targeted query for this specific subsection
            sub_query = f"{title} - {sub_title}"
            if keywords:
                # Include up to 5 keywords for boosting
                sub_query += f". Keywords: {', '.join(keywords[:5])}"
            
            # Smaller retrieval: 5 candidates → rerank → keep 3
            sub_chunks = self.rag.retrieve_and_rerank(sub_query, initial_k=5, final_k=3)
            
            subsection_contents.append(
                SubsectionContent(
                    title=sub_title,
                    content=self._extract_subsection_content(sub_title, sub_chunks),
                    chunks=sub_chunks,
                )
            )

        # ── Package the results ──────────────────────────────────────────
        section_content = SectionContent(
            section_key=section_key,
            title=title,
            description=description,
            summary=summary,
            subsections=subsection_contents,
            all_chunks=chunks,
        )
        
        # Store for later access via get_section()
        self.extracted_sections[section_key] = section_content
        print(f"   ✅ {title} — {len(chunks)} chunks, {len(subsections)} subsections")
        return section_content

    @staticmethod
    def _extract_subsection_content(subsection_title: str,
                                    chunks: List[Dict]) -> str:
        """
        Combine top chunk texts for a subsection into a brief summary.
        
        This is a simple concatenation (not LLM-based) — we take the text
        from the top 2 chunks and truncate to 500 characters. Good enough
        for a quick preview; the full LLM summary is in section.summary.
        
        Args:
            subsection_title: Name of the subsection
            chunks: Retrieved chunks for this subsection
        
        Returns:
            Combined text string, or "Not found in document" if no chunks
        """
        if not chunks:
            return "Not found in document"
        # Take text from the top 2 chunks and join them
        parts = [c["text"] for c in chunks[:2]]
        return " ".join(parts)[:500]  # Truncate to 500 chars for preview

    # =====================================================================
    # EXTRACT ALL SECTIONS
    # =====================================================================
    def extract_all_sections(self) -> Dict[str, SectionContent]:
        """
        Extract every discovered section one by one.
        
        Iterates through all section keys in self.sections (from discovery)
        and calls extract_section() for each one.
        
        Returns:
            Dict mapping section_key → SectionContent
        """
        n = len(self.sections)
        print(f"\n📄 Extracting all {n} discovered sections …")
        for section_key in self.sections:
            self.extract_section(section_key)
        print(f"\n✅ Extracted {len(self.extracted_sections)} sections total")
        return self.extracted_sections

    # =====================================================================
    # ACCESSORS — For retrieving extracted content
    # =====================================================================
    def get_section(self, section_key: str) -> Optional[SectionContent]:
        """Get the extracted content for a specific section (None if not yet extracted)."""
        return self.extracted_sections.get(section_key)

    def get_chunks_for_section(self, section_key: str) -> List[Dict]:
        """Get all source chunks used for a section's extraction."""
        sec = self.extracted_sections.get(section_key)
        return sec.all_chunks if sec else []

    def get_chunks_for_subsection(self, section_key: str,
                                  subsection_index: int) -> List[Dict]:
        """Get the source chunks for a specific subsection by index."""
        sec = self.extracted_sections.get(section_key)
        if sec and 0 <= subsection_index < len(sec.subsections):
            return sec.subsections[subsection_index].chunks
        return []

    def get_navigation_structure(self) -> List[Dict]:
        """
        Build a navigation tree suitable for a UI sidebar.
        
        Returns a list of section dicts, each containing:
            - key, title, description, keywords
            - subsections: list of dicts with index, title, has_content, chunks
        
        This is used by the Streamlit app to render the dynamic sidebar.
        The has_content flag indicates whether content was found for that
        subsection (True) or not (False / "Not found in document").
        """
        nav = []
        for section_key, cfg in self.sections.items():
            section_data = {
                "key": section_key,
                "title": cfg["title"],
                "description": cfg.get("description", ""),
                "keywords": cfg.get("keywords", []),
                "subsections": [],
            }
            for idx, sub_title in enumerate(cfg.get("subsections", [])):
                sub_data = {
                    "index": idx,
                    "title": sub_title,
                    "has_content": False,    # Default: no content found
                    "chunks": [],
                }
                # Check if we've already extracted this section and subsection
                extracted = self.extracted_sections.get(section_key)
                if extracted and idx < len(extracted.subsections):
                    sub = extracted.subsections[idx]
                    sub_data["has_content"] = len(sub.chunks) > 0
                    sub_data["chunks"] = sub.chunks
                section_data["subsections"].append(sub_data)
            nav.append(section_data)
        return nav


# ── Verify the class is ready ────────────────────────────────────────────
print("✅ SectionExtractor class defined")
print("   • extract_section()        — RAG + LLM for one section (keyword-boosted)")
print("   • extract_all_sections()   — extract every discovered section sequentially")
print("   • get_navigation_structure() — build sidebar tree for the Streamlit UI")
print("   • Uses dynamically discovered sections — no hardcoded MF_SECTIONS!")

---

## 6 · Run the Pipeline

Now we execute the full pipeline step by step. Each cell below is one stage. All outputs are printed to the terminal.

**Pipeline:**  
`Set PDF path` → `Validate` → `Discover sections` → `Process with Docling` → `Index in ChromaDB` → `Extract sections` → `Chat Q&A`

In [ ]:
# =============================================================================
# STEP 0 — Set the PDF file path
# =============================================================================
# This is the entry point of the pipeline. Point PDF_PATH to your mutual fund
# prospectus/factsheet PDF file. The path is relative to the notebook's
# working directory (the project root).
#
# Supported document types:
#   - Mutual fund factsheets
#   - Scheme Information Documents (SID)
#   - Key Information Memorandums (KIM)
#   - Fund prospectuses
#
# The pipeline will VALIDATE whether the file is actually a mutual fund
# document before processing (Step 1). If validation fails, the pipeline stops.
# =============================================================================

# ── Point this to your PDF file ──────────────────────────────────────────
PDF_PATH = "Fund Facts - HDFC Income Fund - December 2025 [a].pdf"

# ── Verify the file exists before proceeding ─────────────────────────────
if os.path.exists(PDF_PATH):
    print(f"✅ PDF found: {PDF_PATH}")
    
    # Quick info: open the PDF briefly just to count pages
    doc = fitz.open(PDF_PATH)
    print(f"   Pages: {len(doc)}")
    doc.close()
else:
    # If the file isn't found, print a helpful error message
    print(f"❌ PDF not found: {PDF_PATH}")
    print("   Please update PDF_PATH to point to your file.")
    print("   Tip: Use an absolute path if the relative path doesn't work.")

In [ ]:
# =============================================================================
# STEP 1 — Validate the document (Vision-based scoring)
# =============================================================================
# Before investing compute in full extraction, we check if the uploaded PDF
# is actually a mutual fund document. This prevents wasting API calls on
# random PDFs (e.g., someone uploads a tax return or a recipe book).
#
# How it works:
#   1. Instantiate DocumentScanner (creates vision + text LLM clients)
#   2. Call validate_document(PDF_PATH):
#      a. Renders the first 5 pages to PNG images using PyMuPDF
#      b. Sends each image to GPT-4o-mini vision with DOCUMENT_VALIDATION_PROMPT
#      c. Model returns JSON: {is_mf_document: true/false, reason: "..."}
#      d. Score starts at 50, adjusts ±10 per page
#      e. Final score ≥ 65 → document accepted
#
# If the document is REJECTED:
#   - The pipeline stops here (no Docling processing, no RAG indexing)
#   - In the Streamlit app, the API returns 422 with the score details
#   - The user sees a "Document Rejected" badge
#
# Typical scores:
#   - 5/5 MF pages  → 50 + 50 = 100 (strong pass)
#   - 4/5 MF pages  → 50 + 30 = 80 (pass)
#   - 3/5 MF pages  → 50 + 10 = 60 (fail — below 65 threshold)
#   - 0/5 MF pages  → 50 - 50 = 0 (strong rejection)
# =============================================================================

# Create the scanner (initialises vision + text LLM clients)
scanner = DocumentScanner()

# Run the validation pipeline
print("=" * 60)
print("STEP 1: DOCUMENT VALIDATION")
print("=" * 60)

# validate_document returns: (is_valid: bool, score: int, summary: str)
is_valid, score, summary = scanner.validate_document(PDF_PATH)

# Display the result
print(f"\n📊 Result: {'PASSED ✅' if is_valid else 'FAILED ❌'}")
print(f"   Score: {score} / threshold {VALIDATION_THRESHOLD}")
print(f"   Summary: {summary}")

# If the document was rejected, warn the user and stop
if not is_valid:
    print("\n⚠️  Document rejected! The pipeline will not continue.")
    print("   Upload a valid mutual fund prospectus/factsheet to proceed.")
    print("   Tip: Check that the PDF actually contains MF-related content.")

In [ ]:
# =============================================================================
# STEP 2 — Discover sections dynamically (Vision + Sliding-Window Memory)
# =============================================================================
# This is the KEY INNOVATION of our pipeline. Instead of hardcoding which
# sections a mutual fund document should have (old approach: MF_SECTIONS dict),
# we let the vision model discover them automatically from the PDF.
#
# Phase 1 — Per-page scanning:
#   For each page in the PDF:
#   1. Render the page to a PNG image (via PyMuPDF)
#   2. Build a prompt that includes the sliding-window memory context
#      (what sections were found on the previous 3 pages — prevents duplicates,
#      detects continuations across page boundaries)
#   3. Send image + prompt to GPT-4o-mini vision
#   4. Model returns JSON: sections_found, subsections, keywords on this page
#   5. Store the model's response in SlidingWindowMemory for the next page
#
# Phase 2 — Consolidation:
#   After ALL pages are scanned, we have N separate page-level JSONs.
#   The text LLM (SECTION_CONSOLIDATION_PROMPT) merges them:
#   - Sections spanning pages 3-5 become ONE entry with pages=[3,4,5]
#   - Duplicate subsections are removed
#   - Keywords from all pages are combined
#   - A unique section_key is generated for each section
#
# Output: A dict like:
#   {"investment_objective": {title, description, subsections, keywords, pages}, ...}
#
# This replaces the old hardcoded MF_SECTIONS dictionary entirely!
# =============================================================================

print("=" * 60)
print("STEP 2: DYNAMIC SECTION DISCOVERY")
print("=" * 60)

# discover_sections() runs both phases internally and returns the consolidated dict
discovered_sections = scanner.discover_sections(PDF_PATH)

# ── Print the discovered structure in a readable format ──────────────────
print("\n" + "=" * 60)
print("DISCOVERED SECTION STRUCTURE")
print("=" * 60)
for key, sec in discovered_sections.items():
    print(f"\n🔹 [{key}] {sec['title']}")
    print(f"   Description: {sec['description']}")
    if sec.get('subsections'):
        print(f"   Subsections:")
        for sub in sec['subsections']:
            print(f"      • {sub}")
    if sec.get('keywords'):
        # Show first 8 keywords (some sections have many)
        print(f"   Keywords: {', '.join(sec['keywords'][:8])}")
    if sec.get('pages'):
        print(f"   Pages: {sec['pages']}")

In [ ]:
# =============================================================================
# STEP 3 — Process the PDF with Docling (chunking + metadata extraction)
# =============================================================================
# Now that we've validated the document and discovered its sections,
# we need to extract the actual text content for RAG indexing.
#
# Docling vs. simple text extraction:
#   - PyPDF2/pdfplumber: Simple text extraction, loses structure
#   - Docling: ML-based layout analysis → reading order, element types,
#     table detection, heading hierarchy, bounding boxes
#
# Why bounding boxes matter:
#   - When a user clicks on a retrieved chunk in the UI, we highlight
#     its position on the PDF page using the bounding box coordinates
#   - Without bbox, we could show the text but not WHERE it appears
#
# Processing pipeline:
#   1. Docling DocumentConverter parses the PDF:
#      - Layout analysis (ML model identifies text blocks, tables, headings)
#      - Reading order determination (which block comes first)
#      - Table structure recognition (rows, columns, cells)
#      - OCR if needed (for scanned documents)
#   2. HybridChunker splits the structured document into semantic chunks:
#      - Respects heading boundaries (doesn't split across headings)
#      - Respects table structures (keeps tables intact if possible)
#      - Max 512 tokens per chunk (configurable)
#      - merge_peers=True: merges small adjacent elements
#   3. For each chunk, we extract metadata:
#      - page_number: which page (for navigation)
#      - bbox: bounding box (for highlighting)
#      - element_type: 'text', 'table', 'heading', etc.
#      - headings: section heading trail (for context)
#
# Note: Docling's first run downloads ~2GB of ML models. Subsequent runs
# use cached models and are much faster.
# =============================================================================

print("=" * 60)
print("STEP 3: DOCUMENT PROCESSING (Docling)")
print("=" * 60)

# Create the processor (lazy-loads Docling components on first use)
processor = DocumentProcessor()

# Run the full processing pipeline: PDF → structured document → chunks
chunks = processor.process_document(PDF_PATH)

# ── Print sample chunks to verify the output ─────────────────────────────
print(f"\n📦 Total chunks: {len(chunks)}")
print("\n--- Sample Chunks (first 5) ---")
for chunk in chunks[:5]:
    print(f"\n   Chunk {chunk.chunk_id}:")
    print(f"   Page: {chunk.page_number}")           # Which page
    print(f"   Type: {chunk.element_type}")           # text/table/heading
    print(f"   Headings: {chunk.headings}")           # Section trail
    print(f"   BBox: {chunk.bbox}")                   # Highlight coordinates
    print(f"   Text: {chunk.text[:120]}…")            # Preview (first 120 chars)

# ── Show chunk distribution across pages ─────────────────────────────────
# This helps verify that all pages were processed (no missing pages)
page_counts = {}
for c in chunks:
    page_counts[c.page_number] = page_counts.get(c.page_number, 0) + 1
print(f"\n📊 Chunks per page:")
for page, count in sorted(page_counts.items()):
    print(f"   Page {page}: {count} chunks")

In [ ]:
# =============================================================================
# STEP 4 — Index chunks in ChromaDB (vector store)
# =============================================================================
# This step creates the vector store that powers all RAG retrieval.
#
# What happens:
#   1. RAGEngine initializes OpenAI Embeddings + ChatOpenAI instances
#   2. index_chunks() converts each ChunkWithMetadata → LangChain Document
#   3. Each Document's text is embedded using text-embedding-3-small
#      (1536-dimensional vectors, ~$0.02 per 1M tokens)
#   4. Vectors + metadata are stored in ChromaDB (persisted to ./chroma_db)
#
# ChromaDB persistence:
#   - Vectors are saved to disk in ./chroma_db/
#   - If you restart the kernel, you don't need to re-embed
#   - However, in this notebook we always re-index for freshness
#
# After indexing, we run a quick test to verify retrieval works:
#   - Search for "investment objective" (a common MF section)
#   - Should return chunks from the relevant pages
#
# Cost estimate: ~200 chunks × 512 tokens ≈ 100K tokens ≈ $0.002 for embedding
# =============================================================================

print("=" * 60)
print("STEP 4: RAG INDEXING (ChromaDB)")
print("=" * 60)

# Create the RAG engine (initializes embedding model + LLM)
rag_engine = RAGEngine()

# Index all chunks into ChromaDB
# This embeds all chunk texts and stores them as vectors
rag_engine.index_chunks(chunks)

# ── Quick test: verify retrieval works ───────────────────────────────────
# Run a simple similarity search to make sure the vector store is functional
print("\n--- Quick Retrieval Test ---")
test_query = "What is the investment objective of this fund?"
test_results = rag_engine.retrieve(test_query, top_k=3)  # Stage 1 only (no reranking)
print(f"   Query: '{test_query}'")
print(f"   Retrieved {len(test_results)} chunks:")
for i, doc in enumerate(test_results):
    page = doc.metadata.get('page_number', '?')
    print(f"   [{i+1}] Page {page}: {doc.page_content[:100]}…")

In [ ]:
# =============================================================================
# STEP 5 — Test Re-ranking (Retrieve → LLM Score → Top results)
# =============================================================================
# This step demonstrates the two-stage retrieval pipeline that makes our
# RAG system better than basic similarity search:
#
# Stage 1 — Vector Similarity Search (fast, broad):
#   - Embed the query using the same embedding model
#   - Find top 10 most similar vectors in ChromaDB (cosine distance)
#   - Fast (~100ms) but can include false positives
#     (e.g., "risk factors" might retrieve "risk-o-meter" chunks)
#
# Stage 2 — LLM Re-ranking (accurate, precise):
#   - Send all 10 chunks to GPT-4o-mini with RERANK_PROMPT
#   - Model scores each chunk 0-10 for relevance to the query:
#     10 = directly answers the query
#     7-9 = highly relevant
#     4-6 = somewhat relevant
#     1-3 = marginally relevant
#     0 = not relevant
#   - Sort by score descending and keep top 5
#
# Why this matters:
#   - Embedding similarity is a blunt instrument — it measures semantic
#     distance, not actual relevance to answering a question
#   - The LLM can understand nuance: "this chunk mentions risk but doesn't
#     actually discuss risk FACTORS" → low score
#   - Result: much higher precision in the final context window
# =============================================================================

print("=" * 60)
print("STEP 5: RE-RANKING DEMONSTRATION")
print("=" * 60)

# Test query about risk factors — a topic that appears in most MF documents
rerank_query = "What are the risk factors associated with this fund?"
print(f"\nQuery: '{rerank_query}'")

# Run the full two-stage pipeline: retrieve (10) → rerank → keep (5)
reranked_results = rag_engine.retrieve_and_rerank(rerank_query)

# Display the re-ranked results with their scores
print(f"\nTop {len(reranked_results)} re-ranked results:")
print("-" * 50)
for i, result in enumerate(reranked_results):
    print(f"\n   [{i+1}] Score: {result['score']}/10")         # LLM relevance score
    print(f"       Page: {result['page_number']}")              # Source page
    print(f"       Type: {result['element_type']}")             # text/table/heading
    print(f"       Headings: {result['headings']}")             # Section trail
    print(f"       Text: {result['text'][:150]}…")              # Preview

In [ ]:
# =============================================================================
# STEP 6 — Extract ALL discovered sections (RAG + LLM summarisation)
# =============================================================================
# This is where everything comes together. For each dynamically discovered
# section, the SectionExtractor:
#
#   1. Builds a rich search query:
#      Query = "{title}: {description}. Looking for: {subsections}. Key terms: {keywords}"
#      Example: "Risk Factors: Types of risks associated with the fund.
#                Looking for: Market Risk, Credit Risk. Key terms: volatility, NAV"
#
#   2. RAG retrieval with keyword boosting (initial_k=15, final_k=10):
#      - The keywords discovered by the vision scanner are appended to
#        the query to improve embedding similarity matching
#      - We retrieve 15 candidates (wider net than usual)
#      - LLM re-ranks and keeps top 10 (more context for sections)
#
#   3. LLM summarisation (SECTION_EXTRACTION_PROMPT):
#      - All 10 chunks are sent to GPT-4o-mini
#      - Model organises the information by subsection
#      - Produces a structured summary with key figures and data points
#
#   4. Per-subsection focused retrieval:
#      - For each subsection (e.g., "Market Risk" under "Risk Factors"),
#        do a separate smaller retrieval (5→3 chunks)
#      - This provides granular content for each sub-topic
#      - Allows the UI to show subsection-specific sources
#
# The result is stored in extractor.extracted_sections keyed by section_key.
# Each entry is a SectionContent dataclass with summary, subsections, chunks.
#
# Cost estimate: ~10 sections × (15 chunks × reranking + extraction)
#                ≈ 20 LLM calls ≈ ~100K tokens ≈ $0.015
# =============================================================================

print("=" * 60)
print("STEP 6: SECTION EXTRACTION")
print("=" * 60)

# Create the extractor with the dynamically discovered sections.
# Note: discovered_sections came from Step 2, NOT from a hardcoded dict.
extractor = SectionExtractor(rag_engine, discovered_sections)

# Extract all sections (this calls extract_section() for each one)
extracted = extractor.extract_all_sections()

# ── Print the extraction results ─────────────────────────────────────────
print("\n" + "=" * 60)
print("EXTRACTION RESULTS")
print("=" * 60)
for key, section in extracted.items():
    print(f"\n{'─' * 50}")
    print(f"📌 {section.title}")
    print(f"{'─' * 50}")
    print(f"   Key: {section.section_key}")           # The unique section identifier
    print(f"   Description: {section.description}")    # What the section covers
    print(f"   Chunks used: {len(section.all_chunks)}")  # How many RAG chunks were used
    print(f"   Subsections: {len(section.subsections)}")  # How many sub-topics
    
    # Print the LLM-generated summary (first 400 chars as preview)
    print(f"\n   Summary (preview):")
    for line in section.summary[:400].split('\n'):
        print(f"   {line}")
    if len(section.summary) > 400:
        print(f"   … (truncated, {len(section.summary)} chars total)")
    
    # Print each subsection with its content and source chunks
    if section.subsections:
        print(f"\n   Subsections:")
        for sub in section.subsections:
            # ✅ = content found, ❌ = not found in document
            has_content = "✅" if sub.content != "Not found in document" else "❌"
            print(f"      {has_content} {sub.title}")
            print(f"         Chunks: {len(sub.chunks)}")         # How many focused chunks
            print(f"         Content: {sub.content[:100]}…")     # Preview of extracted text

## 7 · Chatbot Q&A — Ask Questions About the Document

The chatbot uses the full RAG pipeline:
1. **Retrieve** top 10 chunks by similarity search
2. **Re-rank** with LLM scoring → keep top 5
3. **Generate** answer using GPT-4o-mini with the re-ranked chunks as context

Each answer includes the source chunks used, so you can verify where the information came from.

In [ ]:
# =============================================================================
# CHATBOT — Q&A with retrieved + re-ranked context (single question demo)
# =============================================================================
# The chatbot uses the full RAG pipeline for each question:
#
#   1. User asks a question (e.g., "What is the investment objective?")
#   2. RETRIEVE: Embed the query → cosine similarity search in ChromaDB → top 10 chunks
#   3. RE-RANK:  Send 10 chunks to GPT-4o-mini → LLM scores each 0-10 → keep top 5
#   4. GENERATE: Build context from the 5 best chunks → GPT-4o-mini generates answer
#   5. Return: (answer_text, source_chunks) — the answer + where it came from
#
# The source_chunks list is crucial for traceability — you can verify
# which pages/chunks the answer was derived from. This is a key differentiator
# from plain ChatGPT: our answers are GROUNDED in the actual document.
#
# The CHAT_SYSTEM_PROMPT includes strict instructions:
#   - ONLY answer from provided chunks (no hallucination)
#   - Say "I couldn't find this" if info not in chunks
#   - Cite page numbers when possible
#   - Be precise with numbers, percentages, dates
# =============================================================================

print("=" * 60)
print("CHATBOT Q&A — Single Question Demo")
print("=" * 60)

# ── Ask a single question to demonstrate the full pipeline ───────────────
question_1 = "What is the investment objective of this fund?"
print(f"\n💬 Q: {question_1}")

# chat() runs: retrieve (10) → rerank (→5) → generate answer
answer_1, sources_1 = rag_engine.chat(question_1)

# Display the answer
print(f"\n🤖 A: {answer_1}")

# Display the source chunks that were used to generate the answer
# This is the "traceability" — you can verify every claim
print(f"\n   📎 Sources used: {len(sources_1)} chunks")
for i, src in enumerate(sources_1):
    print(f"   [{i+1}] Page {src['page_number']} (score {src['score']}): {src['text'][:80]}…")

In [ ]:
# =============================================================================
# BATCH CHATBOT QUERIES — Ask multiple questions in sequence
# =============================================================================
# This cell runs several common mutual fund questions through the chatbot.
# Each question goes through the FULL pipeline independently:
#   embed → retrieve (10) → rerank (→5) → generate
#
# These questions cover typical information an investor would look for:
#   1. Expense ratio      — cost of investing in the fund
#   2. Fund manager       — who manages the fund and their experience
#   3. Asset allocation   — how the fund distributes investments
#   4. Exit load          — fees for early withdrawal
#   5. Benchmark          — index the fund is measured against
#
# You can modify this list to ask your own questions!
# Each answer is independently grounded in retrieved chunks.
# =============================================================================

# Define a batch of questions to ask
questions = [
    "What is the expense ratio of this fund?",
    "Who is the fund manager and what is their experience?",
    "What is the asset allocation of this fund?",
    "What are the exit load charges?",
    "What benchmark does this fund track?",
]

# Process each question through the full RAG pipeline
for idx, question in enumerate(questions, 1):
    print(f"\n{'═' * 60}")
    print(f"💬 Q{idx}: {question}")
    print(f"{'═' * 60}")
    
    # Full pipeline: retrieve → rerank → generate
    answer, sources = rag_engine.chat(question)
    
    # Print the answer
    print(f"\n🤖 Answer:\n{answer}")
    
    # Show which pages the answer came from (for quick verification)
    print(f"\n   📎 Sources: {len(sources)} chunks from pages: ", end="")
    print(", ".join(str(s['page_number']) for s in sources))

## 8 · Interactive Chat Loop

Run the cell below to start an interactive chat session. Type your questions and get answers from the document. Type `quit` or `exit` to stop.

In [ ]:
# =============================================================================
# INTERACTIVE CHAT — Ask your own questions in a loop
# =============================================================================
# This cell starts an interactive input loop where you can type your own
# questions about the mutual fund document and get answers in real time.
#
# Each question goes through the FULL RAG pipeline:
#   1. Your question → embedded as a 1536-dim vector
#   2. ChromaDB cosine similarity search → top 10 candidate chunks
#   3. GPT-4o-mini re-ranks → scores each 0-10 → keeps top 5
#   4. Top 5 chunks injected as context into CHAT_SYSTEM_PROMPT
#   5. GPT-4o-mini generates an answer grounded in those chunks
#
# Tips for good questions:
#   - Be specific: "What is the expense ratio?" > "Tell me about fees"
#   - Ask about numbers: "What is the AUM?" → precise answer
#   - Ask about structure: "What sections does this document have?"
#
# Type 'quit', 'exit', or 'q' to stop the loop and continue to the next cell.
#
# NOTE: In Jupyter notebooks, input() blocks the kernel. If the cell seems
# stuck, make sure you've typed a question in the input box that appears.
# =============================================================================

print("=" * 60)
print("INTERACTIVE CHAT — Ask questions about the document")
print("Type 'quit' or 'exit' to stop")
print("=" * 60)

while True:
    # Get user input (blocks until the user types something)
    user_query = input("\n💬 Your question: ").strip()
    
    # Check for exit commands
    if user_query.lower() in ('quit', 'exit', 'q'):
        print("\n👋 Chat ended. Moving on to the next cell.")
        break
    
    # Skip empty inputs
    if not user_query:
        print("   (empty question, try again)")
        continue
    
    # Run the full RAG pipeline: retrieve → rerank → generate
    answer, sources = rag_engine.chat(user_query)
    
    # Display the answer
    print(f"\n🤖 Answer:\n{answer}")
    
    # Show source pages for traceability
    print(f"\n   📎 Sources: {len(sources)} chunks from pages: ", end="")
    print(", ".join(str(s['page_number']) for s in sources))

## 9 · Inspect a Specific Section

Pick a specific discovered section and inspect its full extracted content — summary, subsections, and source chunks.

In [ ]:
# =============================================================================
# INSPECT A SECTION — View full details of one extracted section
# =============================================================================
# This cell lets you deep-dive into a specific section's extracted content.
# It prints:
#   1. The section's description (from discovery)
#   2. The full LLM-generated summary (from SECTION_EXTRACTION_PROMPT)
#   3. Each subsection with its extracted content and source chunks
#   4. All source chunks with their re-rank scores and pages
#
# How to use:
#   - The cell defaults to the FIRST discovered section
#   - To inspect a different section, change `target_key` below
#     to any key from the "Available sections" list
#   - Example: target_key = "risk_factors"
#
# This is equivalent to what the Streamlit UI shows in the "Sections" tab,
# but in raw text form for debugging and inspection.
# =============================================================================

# ── List all available section keys ──────────────────────────────────────
print("Available sections:")
for i, (key, sec) in enumerate(discovered_sections.items()):
    print(f"   {i+1}. {key} → {sec['title']}")

# ── Pick a section to inspect ────────────────────────────────────────────
# Change this to inspect a different section (copy a key from the list above)
target_key = list(discovered_sections.keys())[0]  # Default: first section
section = extractor.get_section(target_key)

if section:
    # ── Section header and description ───────────────────────────────
    print(f"\n{'═' * 60}")
    print(f"📌 SECTION: {section.title}")
    print(f"{'═' * 60}")
    print(f"\nDescription: {section.description}")
    
    # ── Full LLM-generated summary ───────────────────────────────────
    # This is the structured output from SECTION_EXTRACTION_PROMPT —
    # the LLM organised the retrieved chunks into a readable report
    print(f"\n--- Full Summary ---")
    print(section.summary)
    
    # ── Per-subsection details ───────────────────────────────────────
    # Each subsection had its own focused retrieval (5→3 chunks)
    print(f"\n--- Subsections ({len(section.subsections)}) ---")
    for sub in section.subsections:
        print(f"\n   ▸ {sub.title}")
        print(f"     Content: {sub.content}")              # Concatenated top chunks
        print(f"     Source chunks: {len(sub.chunks)}")     # How many focused chunks
        for c in sub.chunks:
            # Show each source chunk with its re-rank score and page
            print(f"       • Page {c['page_number']} (score {c['score']}): {c['text'][:80]}…")
    
    # ── All source chunks used for the section ───────────────────────
    # These are the chunks that went into the SECTION_EXTRACTION_PROMPT
    print(f"\n--- All Source Chunks ({len(section.all_chunks)}) ---")
    for c in section.all_chunks:
        print(f"   Page {c['page_number']} | Score {c['score']} | {c['element_type']}")
        print(f"   {c['text'][:120]}…")
        print()
else:
    print(f"Section '{target_key}' not yet extracted. Run Step 6 first.")

## 10 · Navigation Structure & Tables

View the full navigation tree (as the UI sidebar would show it) and any tables detected in the document.

In [ ]:
# =============================================================================
# NAVIGATION STRUCTURE — How the UI sidebar is built
# =============================================================================
# The Streamlit app shows a sidebar with all discovered sections and their
# subsections. This cell displays the same navigation tree structure that
# the UI renders, so you can see exactly what the user would see.
#
# The navigation tree shows:
#   📁 Section Title
#      ✅ Subsection 1 (N chunks)    ← content was found
#      ⬜ Subsection 2 (0 chunks)    ← no content found in document
#
# The has_content flag is determined by whether focused subsection retrieval
# returned any chunks. ⬜ doesn't necessarily mean the info isn't in the
# document — it might mean the search didn't find relevant chunks for that
# specific subsection title.
# =============================================================================

# Build the navigation tree structure
nav = extractor.get_navigation_structure()

print("=" * 60)
print("NAVIGATION TREE (as the Streamlit UI sidebar would show)")
print("=" * 60)
for section in nav:
    print(f"\n📁 {section['title']}")
    print(f"   Key: {section['key']}")              # The section_key identifier
    if section.get('keywords'):
        # Show top 6 keywords (used for keyword-boosted retrieval)
        print(f"   🔑 {', '.join(section['keywords'][:6])}")
    for sub in section.get('subsections', []):
        # ✅ = chunks found, ⬜ = no chunks found for this subsection
        status = "✅" if sub['has_content'] else "⬜"
        print(f"   {status} {sub['title']} ({len(sub['chunks'])} chunks)")

In [ ]:
# =============================================================================
# TABLES — Extract tables detected by Docling
# =============================================================================
# Docling's layout analysis model can detect and extract tables from PDFs.
# Each detected table is exported as a pandas DataFrame.
#
# This is useful for:
#   - Viewing portfolio holdings tables
#   - Seeing performance/returns data
#   - Checking asset allocation breakdowns
#   - Examining expense ratio details
#
# Each table includes:
#   - dataframe: The table content as a pandas DataFrame
#   - page_number: Which page the table appears on
#   - bbox: Bounding box coordinates for highlighting
# =============================================================================

# Get all tables detected by Docling during document processing
tables = processor.get_tables()
print(f"\n{'=' * 60}")
print(f"TABLES FOUND: {len(tables)}")
print(f"{'=' * 60}")

for i, table in enumerate(tables):
    print(f"\n📊 Table {i+1} (Page {table['page_number']})")
    print(f"   BBox: {table['bbox']}")            # Position on the page
    df = table['dataframe']
    print(f"   Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"   Columns: {list(df.columns)}")
    # Display the table (max 5 rows to keep output manageable)
    print(df.to_string(index=False, max_rows=5))
    print()

---

## Summary

| Step | What it does | Key class/method |
|------|-------------|-----------------|
| 1 | **Configuration** | All settings + 6 prompts |
| 2 | **Validation** | `DocumentScanner.validate_document()` — vision scoring ±10 |
| 3 | **Section Discovery** | `DocumentScanner.discover_sections()` — vision + sliding memory |
| 4 | **Docling Processing** | `DocumentProcessor.process_document()` — PDF → chunks with bbox |
| 5 | **RAG Indexing** | `RAGEngine.index_chunks()` — embed → ChromaDB |
| 6 | **Re-ranking** | `RAGEngine.retrieve_and_rerank()` — similarity → LLM score |
| 7 | **Section Extraction** | `SectionExtractor.extract_all_sections()` — RAG + LLM summaries |
| 8 | **Chat Q&A** | `RAGEngine.chat()` — retrieve → rerank → generate |

All sections are **dynamically discovered** — no hardcoded section names.  
The **Streamlit + FastAPI** application (in `application/`) uses these same classes.